In [52]:
# plot_roll_csv.py
# -----------------------------------------------------------------------------
# Plot and analyze gyro-only roll logs from your ESP32 (LSM9DS1) CSV dump.
#
# What this script does:
#  1) Loads roll.csv (even if it contains miniterm banners / --- CSV DUMP ... --- lines)
#  2) Estimates the gyroscope roll-rate bias (offset) from an initial "still" window
#  3) Plots:
#       - roll_gyro_deg vs time
#       - omega_roll_deg_s and gx/gy/gz vs time
#       - Histogram of omega_roll_deg_s in the bias window
#       - (Important) A *proper* "debiased roll" computed using the SAME dt samples
#
# Why "proper debiased roll" matters:
#  Your firmware integrates only when a gyro sample is actually read:
#      theta[k+1] = theta[k] + omega[k] * dt[k]
#  So to remove bias consistently, do:
#      theta_debiased[k] = Σ_{i=1..k} (omega[i] - b_hat) * dt[i]
#
#  where b_hat is the estimated constant bias (deg/s) from an initial stationary window.
#
# Usage:
#   python3 plot_roll_csv.py /path/to/roll.csv
#
# Dependencies:
#   pip install pandas numpy plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


# -----------------------------------------------------------------------------
# User settings (adjust as needed)
# -----------------------------------------------------------------------------
BIAS_WINDOW_S = 20.0   # seconds used to estimate bias (keep IMU still for this duration)
SKIP_FIRST_S = 0.5     # seconds to ignore at start (startup transient)
TEMPLATE = "plotly_white"

# A clean, readable palette (Plotly will cycle through these in order)
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: remove obvious outliers in bias window (helps if you bumped the sensor)
# Set to None to disable. Typical values: 3.0 to 5.0
BIAS_CLIP_STD = 4.0


# -----------------------------------------------------------------------------
# CSV loading helper
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Loads the CSV while ignoring miniterm banners / dump markers / accidental plotter lines.

    Your serial dump can contain lines like:
      --- Miniterm on /dev/ttyUSB0 ...
      --- CSV DUMP START ---
      --- CSV DUMP END ---
    and sometimes "roll_gyro_deg:xx" if you accidentally captured plotter output.

    We strip those and parse only actual CSV rows.
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    cleaned_lines: list[str] = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        # Skip any banner/marker lines
        if s.startswith("---"):
            continue
        # Skip plotter-only format lines (if accidentally captured)
        if s.startswith("roll_gyro_deg:"):
            continue
        cleaned_lines.append(ln)

    df = pd.read_csv(StringIO("".join(cleaned_lines)))

    expected = [
        "time_s", "dt_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "omega_roll_deg_s",
        "roll_gyro_deg",
    ]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}\nFound: {list(df.columns)}")

    # Sort and reset index for safety
    df = df.sort_values("time_s").reset_index(drop=True)

    # Ensure numeric types
    for c in expected:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop rows with NaNs in critical columns
    df = df.dropna(subset=["time_s", "dt_s", "omega_roll_deg_s", "roll_gyro_deg"]).reset_index(drop=True)

    return df


# -----------------------------------------------------------------------------
# Bias (offset) estimation
# -----------------------------------------------------------------------------
def estimate_bias(df: pd.DataFrame) -> dict:
    """
    Estimate gyro roll-rate bias b_hat from an initial window where the IMU is still.

    We assume the measurement model (for roll rate):
        omega_meas(t) = omega_true(t) + b + noise

    When the IMU is still (omega_true ≈ 0), then:
        omega_meas ≈ b + noise
    So a good bias estimate is the mean of omega_meas over a stationary window.

    Returns stats including:
        bias_mean_deg_s, bias_median_deg_s, bias_std_deg_s,
        drift_deg_per_min, predicted_drift_60s
    """
    t0 = float(df["time_s"].min())
    mask = (df["time_s"] >= t0 + SKIP_FIRST_S) & (df["time_s"] <= t0 + BIAS_WINDOW_S)

    w = df.loc[mask, "omega_roll_deg_s"].to_numpy()
    if w.size < 10:
        raise ValueError(
            "Not enough samples in the bias window.\n"
            "Increase BIAS_WINDOW_S or check that your CSV contains data early on."
        )

    # Optional robust clipping (remove occasional spikes if you moved the board)
    if BIAS_CLIP_STD is not None and w.size >= 20:
        mu = np.mean(w)
        sigma = np.std(w, ddof=1) if w.size > 1 else 0.0
        if sigma > 1e-12:
            keep = np.abs(w - mu) <= (BIAS_CLIP_STD * sigma)
            w = w[keep]

    bias_mean = float(np.mean(w))
    bias_median = float(np.median(w))
    bias_std = float(np.std(w, ddof=1)) if w.size > 1 else 0.0

    drift_deg_per_min = bias_mean * 60.0
    predicted_drift_60s = bias_mean * 60.0

    return {
        "bias_mean_deg_s": bias_mean,
        "bias_median_deg_s": bias_median,
        "bias_std_deg_s": bias_std,
        "drift_deg_per_min": drift_deg_per_min,
        "predicted_drift_60s_deg": predicted_drift_60s,
        "n_bias_samples": int(w.size),
        "bias_window_used_s": float(BIAS_WINDOW_S),
        "skip_first_s": float(SKIP_FIRST_S),
    }


# -----------------------------------------------------------------------------
# Proper debiased integration using recorded dt_s
# -----------------------------------------------------------------------------
def compute_debiased_roll(df: pd.DataFrame, bias_deg_s: float) -> np.ndarray:
    """
    Compute a *proper* debiased roll estimate using the *same dt_s samples* you integrated.

    Your firmware effectively does:
        theta[k] = theta[k-1] + omega[k] * dt[k]

    If omega[k] = omega_true[k] + b + noise, then integrating omega causes drift b*t.
    To remove bias consistently with your discrete integration, do:

        theta_debiased[k] = Σ_{i=1..k} (omega[i] - b_hat) * dt[i]

    This uses dt[i] from the CSV, so it remains consistent even if the IMU sampling
    is not perfectly uniform.
    """
    omega = df["omega_roll_deg_s"].to_numpy()
    dt = df["dt_s"].to_numpy()

    # elementwise: (omega - bias) * dt  -> deg
    increments = (omega - bias_deg_s) * dt
    theta_debiased = np.cumsum(increments)

    return theta_debiased


# -----------------------------------------------------------------------------
# Plotting helpers
# -----------------------------------------------------------------------------
def plot_roll_timeseries(df: pd.DataFrame, theta_debiased: np.ndarray):
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"],
        mode="lines", name="roll_gyro_deg (firmware integration)",
        line=dict(width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=theta_debiased,
        mode="lines", name="roll_debiased (Σ (ω - b̂) dt)",
        line=dict(width=2, dash="dash")
    ))

    fig.update_layout(
        title="Gyro-only Roll vs Time (Integrated) + Proper Debiased Roll",
        xaxis_title="time (s)",
        yaxis_title="roll (deg)",
        hovermode="x unified",
        legend_title="signals",
    )
    fig.show()


def plot_gyro_rates(df: pd.DataFrame):
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    for col in ["gx_deg_s", "gy_deg_s", "gz_deg_s", "omega_roll_deg_s"]:
        fig.add_trace(go.Scatter(
            x=df["time_s"], y=df[col],
            mode="lines", name=col,
            line=dict(width=2)
        ))

    fig.update_layout(
        title="Gyroscope Angular Rates vs Time",
        xaxis_title="time (s)",
        yaxis_title="angular rate (deg/s)",
        hovermode="x unified",
        legend_title="channels",
    )
    fig.show()


def plot_bias_window_hist(df: pd.DataFrame):
    t0 = float(df["time_s"].min())
    mask = (df["time_s"] >= t0 + SKIP_FIRST_S) & (df["time_s"] <= t0 + BIAS_WINDOW_S)
    w = df.loc[mask, "omega_roll_deg_s"]

    fig = px.histogram(
        w,
        nbins=50,
        template=TEMPLATE,
        title=f"Bias Window Histogram: omega_roll_deg_s (first {BIAS_WINDOW_S}s, skip {SKIP_FIRST_S}s)"
    )
    fig.update_layout(
        xaxis_title="omega_roll_deg_s (deg/s)",
        yaxis_title="count",
    )
    fig.show()


def plot_dt(df: pd.DataFrame):
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["dt_s"],
        mode="lines", name="dt_s",
        line=dict(width=2)
    ))

    fig.update_layout(
        title="Recorded dt_s vs Time (Sampling / Integration Step Size)",
        xaxis_title="time (s)",
        yaxis_title="dt (s)",
        hovermode="x unified",
    )
    fig.show()


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    if len(sys.argv) < 2:
        print("Usage: python3 plot_roll_csv.py /path/to/roll.csv")
        sys.exit(1)

    path = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/data/rollbiased0.csv"
    df = load_csv_clean(path)

    # Estimate bias from early stationary window
    stats = estimate_bias(df)
    bias_mean = stats["bias_mean_deg_s"]

    # Proper debiased roll using dt samples
    theta_debiased = compute_debiased_roll(df, bias_deg_s=bias_mean)

    # Print summary
    print("\n=== Gyro Roll-Rate Offset (Bias) Estimate ===")
    print(f"Bias window: first {stats['bias_window_used_s']}s (skipping first {stats['skip_first_s']}s)")
    print(f"Samples used: {stats['n_bias_samples']}")
    print(f"Mean bias      : {stats['bias_mean_deg_s']:+.6f} deg/s")
    print(f"Median bias    : {stats['bias_median_deg_s']:+.6f} deg/s")
    print(f"Std dev (noise): {stats['bias_std_deg_s']:.6f} deg/s")
    print(f"Drift rate (if uncorrected): {stats['drift_deg_per_min']:+.3f} deg/min")
    print(f"Predicted drift over 60s    : {stats['predicted_drift_60s_deg']:+.3f} deg")

    # Extra: compare final values (helps sanity-check)
    print("\n=== End-of-run sanity check ===")
    print(f"Final firmware roll_gyro_deg       : {float(df['roll_gyro_deg'].iloc[-1]):+.3f} deg")
    print(f"Final proper debiased roll (script): {float(theta_debiased[-1]):+.3f} deg")

    # Plots
    plot_roll_timeseries(df, theta_debiased)
    plot_gyro_rates(df)
    plot_bias_window_hist(df)
    plot_dt(df)


if __name__ == "__main__":
    main()



=== Gyro Roll-Rate Offset (Bias) Estimate ===
Bias window: first 20.0s (skipping first 0.5s)
Samples used: 1843
Mean bias      : +1.808436 deg/s
Median bias    : +1.831055 deg/s
Std dev (noise): 0.067425 deg/s
Drift rate (if uncorrected): +108.506 deg/min
Predicted drift over 60s    : +108.506 deg

=== End-of-run sanity check ===
Final firmware roll_gyro_deg       : +49.863 deg
Final proper debiased roll (script): -0.076 deg


In [53]:
# plot_roll_unbiased_csv.py
# -----------------------------------------------------------------------------
# Plot and analyze the "biased + unbiased" roll log produced by your ESP32 code:
#
# CSV columns (from your firmware):
#   time_s, dt_s, omega_meas_deg_s, omega_corr_deg_s, roll_biased_deg, roll_unbiased_deg
#
# Assumption for this script (as you asked):
#   - The IMU was kept STILL for the ENTIRE duration.
#   - Therefore, omega_true ≈ 0 for all time, so omega_meas ≈ bias + noise.
#   - We estimate a single constant bias from the full dataset:
#         b_hat = mean(omega_meas_deg_s)
#
# We then:
#   1) Plot raw omega_meas and omega_corr vs time
#   2) Plot roll_biased and roll_unbiased (firmware) vs time
#   3) Recompute our own unbiased roll using the same dt samples:
#         theta_unbiased_recomputed[k] = Σ (omega_meas[k] - b_hat) * dt[k]
#      (This is the mathematically clean discrete-time bias correction)
#   4) Show histogram of omega_meas (noise distribution around bias)
#
# Usage:
#   python3 plot_roll_unbiased_csv.py /path/to/roll_unbiased0.csv
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


# ----------------------------- Styling -----------------------------
TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]
# ------------------------------------------------------------------


def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Loads the CSV while removing miniterm banners / dump markers, if present.
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    cleaned = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):  # miniterm banner / CSV DUMP markers
            continue
        cleaned.append(ln)

    df = pd.read_csv(StringIO("".join(cleaned)))

    expected = [
        "time_s",
        "dt_s",
        "omega_meas_deg_s",
        "omega_corr_deg_s",
        "roll_biased_deg",
        "roll_unbiased_deg",
    ]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in CSV: {missing}\nFound: {list(df.columns)}")

    # Convert to numeric and drop bad rows
    for c in expected:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=expected).sort_values("time_s").reset_index(drop=True)

    return df


def estimate_bias_full_run(df: pd.DataFrame) -> dict:
    """
    Under the "still for all time" assumption:
      omega_true ≈ 0  -> omega_meas ≈ b + noise
    So b_hat = mean(omega_meas).

    Also compute a noise std estimate around that mean.
    """
    w = df["omega_meas_deg_s"].to_numpy()
    b_hat = float(np.mean(w))
    med = float(np.median(w))
    std = float(np.std(w, ddof=1)) if w.size > 1 else 0.0

    # Drift rate if you integrate the bias (deg/min)
    drift_deg_per_min = b_hat * 60.0

    # Predicted drift over the whole recorded interval:
    # Use total elapsed time from the data itself
    T = float(df["time_s"].iloc[-1] - df["time_s"].iloc[0])
    predicted_drift_deg = b_hat * T

    return {
        "bias_mean_deg_s": b_hat,
        "bias_median_deg_s": med,
        "noise_std_deg_s": std,
        "drift_deg_per_min": drift_deg_per_min,
        "T_total_s": T,
        "predicted_drift_deg": predicted_drift_deg,
        "n_samples": int(w.size),
    }


def recompute_unbiased_roll(df: pd.DataFrame, bias_deg_s: float) -> np.ndarray:
    """
    Recompute an unbiased roll angle using discrete integration with recorded dt:

      theta_unbiased[k] = Σ_{i=1..k} (omega_meas[i] - b_hat) * dt[i]

    This is the clean discrete-time bias correction consistent with your integration model.
    """
    omega = df["omega_meas_deg_s"].to_numpy()
    dt = df["dt_s"].to_numpy()
    return np.cumsum((omega - bias_deg_s) * dt)


def plot_rates(df: pd.DataFrame, bias_deg_s: float):
    """
    Plot omega_meas and omega_corr. Also draw a horizontal line at the estimated bias.
    """
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_meas_deg_s"],
        mode="lines", name="omega_meas_deg_s (raw gyro)",
        line=dict(width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_corr_deg_s"],
        mode="lines", name="omega_corr_deg_s (firmware: omega - bias_const)",
        line=dict(width=2)
    ))

    # Estimated bias line
    fig.add_trace(go.Scatter(
        x=[df["time_s"].iloc[0], df["time_s"].iloc[-1]],
        y=[bias_deg_s, bias_deg_s],
        mode="lines", name=f"b_hat = mean(omega_meas) = {bias_deg_s:.4f} deg/s",
        line=dict(width=2, dash="dash")
    ))

    fig.update_layout(
        title="Gyro Angular Rate vs Time (Still Run)",
        xaxis_title="time (s)",
        yaxis_title="angular rate (deg/s)",
        hovermode="x unified",
        legend_title="signals",
    )
    fig.show()


def plot_rolls(df: pd.DataFrame, roll_unbiased_recomputed: np.ndarray):
    """
    Plot firmware integrated roll (biased & unbiased) and compare with recomputed unbiased roll.
    """
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_biased_deg"],
        mode="lines", name="roll_biased_deg (firmware)",
        line=dict(width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_unbiased_deg"],
        mode="lines", name="roll_unbiased_deg (firmware)",
        line=dict(width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time_s"], y=roll_unbiased_recomputed,
        mode="lines", name="roll_unbiased_recomputed (Σ (ω - b_hat) dt)",
        line=dict(width=2, dash="dash")
    ))

    fig.update_layout(
        title="Integrated Roll vs Time (Still Run)",
        xaxis_title="time (s)",
        yaxis_title="roll (deg)",
        hovermode="x unified",
        legend_title="signals",
    )
    fig.show()


def plot_hist(df: pd.DataFrame, bias_deg_s: float):
    """
    Histogram of omega_meas with a vertical line at bias mean.
    """
    fig = px.histogram(
        df, x="omega_meas_deg_s", nbins=60,
        template=TEMPLATE,
        title="Histogram of omega_meas_deg_s (Still Run)"
    )
    fig.update_layout(xaxis_title="omega_meas_deg_s (deg/s)", yaxis_title="count")

    fig.add_vline(
        x=bias_deg_s,
        line_width=2,
        line_dash="dash",
        annotation_text=f"mean bias = {bias_deg_s:.4f} deg/s",
        annotation_position="top right",
    )
    fig.show()


def plot_dt(df: pd.DataFrame):
    """
    dt sanity plot. For a stable IMU update, dt should hover around a consistent value.
    """
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["dt_s"],
        mode="lines", name="dt_s",
        line=dict(width=2)
    ))
    fig.update_layout(
        title="dt_s vs Time (Gyro Sample Interval)",
        xaxis_title="time (s)",
        yaxis_title="dt (s)",
        hovermode="x unified",
    )
    fig.show()


def main():
    if len(sys.argv) < 2:
        print("Usage: python3 plot_roll_unbiased_csv.py /path/to/roll_unbiased0.csv")
        sys.exit(1)

    path = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/data/roll_unbiased0.csv"
    df = load_csv_clean(path)

    stats = estimate_bias_full_run(df)
    b_hat = stats["bias_mean_deg_s"]

    roll_unbiased_recomputed = recompute_unbiased_roll(df, bias_deg_s=b_hat)

    # ----------------------------- Print summary -----------------------------
    print("\n=== Bias Estimate (Assuming STILL for Entire Run) ===")
    print(f"Samples used            : {stats['n_samples']}")
    print(f"Total duration (s)      : {stats['T_total_s']:.3f}")
    print(f"Mean bias b_hat         : {stats['bias_mean_deg_s']:+.6f} deg/s")
    print(f"Median                  : {stats['bias_median_deg_s']:+.6f} deg/s")
    print(f"Noise std (around mean) : {stats['noise_std_deg_s']:.6f} deg/s")
    print(f"Drift rate if uncorrected: {stats['drift_deg_per_min']:+.3f} deg/min")
    print(f"Predicted drift over run : {stats['predicted_drift_deg']:+.3f} deg")

    print("\n=== End-of-run sanity check ===")
    print(f"Final roll_biased_deg (firmware)        : {float(df['roll_biased_deg'].iloc[-1]):+.3f} deg")
    print(f"Final roll_unbiased_deg (firmware)      : {float(df['roll_unbiased_deg'].iloc[-1]):+.3f} deg")
    print(f"Final roll_unbiased_recomputed (script) : {float(roll_unbiased_recomputed[-1]):+.3f} deg")

    # ----------------------------- Plots -----------------------------
    plot_rates(df, bias_deg_s=b_hat)
    plot_rolls(df, roll_unbiased_recomputed)
    plot_hist(df, bias_deg_s=b_hat)
    plot_dt(df)


if __name__ == "__main__":
    main()



=== Bias Estimate (Assuming STILL for Entire Run) ===
Samples used            : 300
Total duration (s)      : 11.570
Mean bias b_hat         : +1.783447 deg/s
Median                  : +1.770020 deg/s
Noise std (around mean) : 0.068579 deg/s
Drift rate if uncorrected: +107.007 deg/min
Predicted drift over run : +20.634 deg

=== End-of-run sanity check ===
Final roll_biased_deg (firmware)        : +16.071 deg
Final roll_unbiased_deg (firmware)      : -0.292 deg
Final roll_unbiased_recomputed (script) : -0.013 deg


In [54]:
# plot_acc_noise_distribution.py
# -----------------------------------------------------------------------------
# Assumption: the IMU was kept STILL for the entire recording.
# Ground truth roll angle = 0 deg.
#
# This script:
#   1) Loads your accelerometer-only CSV (time_s, dt_s, ax_g, ay_g, az_g, roll_acc_deg)
#   2) Plots roll_acc_deg over time with a ground-truth zero line
#   3) Computes noise = roll_acc_deg - 0
#   4) Plots the noise distribution (histogram + KDE) and prints mean/std
#
# Usage:
#   python3 plot_acc_noise_distribution.py /path/to/roll_acc_only.csv
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: skip first few seconds (startup transients)
SKIP_FIRST_S = 1.0

# Optional: remove outliers beyond N std dev when plotting noise distribution
OUTLIER_CLIP_STD = 5.0


def load_csv_clean(path: str) -> pd.DataFrame:
    """Loads CSV and ignores miniterm banners / dump markers if present."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    cleaned = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):  # miniterm banners / CSV DUMP markers
            continue
        cleaned.append(ln)

    df = pd.read_csv(StringIO("".join(cleaned)))

    expected = ["time_s", "dt_s", "ax_g", "ay_g", "az_g", "roll_acc_deg"]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}\nFound: {list(df.columns)}")

    for c in expected:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=expected).sort_values("time_s").reset_index(drop=True)

    return df


def plot_roll_vs_time(df: pd.DataFrame):
    """Plot roll_acc_deg vs time with ground-truth (0 deg) line."""
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"],
        y=df["roll_acc_deg"],
        mode="lines",
        name="roll_acc_deg (measured)",
        line=dict(width=2),
    ))

    # Ground truth: 0 deg (sensor kept still)
    fig.add_trace(go.Scatter(
        x=[df["time_s"].iloc[0], df["time_s"].iloc[-1]],
        y=[0.0, 0.0],
        mode="lines",
        name="ground truth = 0 deg",
        line=dict(width=2, dash="dash"),
    ))

    fig.update_layout(
        title="Accelerometer-only Roll vs Time (Ground Truth = 0°)",
        xaxis_title="time (s)",
        yaxis_title="roll (deg)",
        hovermode="x unified",
        legend_title="signals",
    )
    fig.show()


def plot_noise_distribution(noise: np.ndarray, mu: float, sigma: float):
    """Plot histogram of noise and annotate mean/std."""
    df_noise = pd.DataFrame({"noise_deg": noise})

    fig = px.histogram(
        df_noise,
        x="noise_deg",
        nbins=60,
        template=TEMPLATE,
        title="Noise Distribution: roll_acc_deg - 0°",
    )
    fig.update_layout(xaxis_title="noise (deg)", yaxis_title="count")

    # Mark mean
    fig.add_vline(
        x=mu,
        line_width=2,
        line_dash="dash",
        annotation_text=f"mean = {mu:.3f}°",
        annotation_position="top right",
    )

    # Mark ±1 std
    fig.add_vline(x=mu + sigma, line_width=1, line_dash="dot")
    fig.add_vline(x=mu - sigma, line_width=1, line_dash="dot")

    fig.show()

    # Also show a box plot for quick spread visualization
    fig2 = px.box(df_noise, y="noise_deg", template=TEMPLATE, title="Noise Spread (Box Plot)")
    fig2.update_layout(yaxis_title="noise (deg)")
    fig2.show()


def main():
    if len(sys.argv) < 2:
        print("Usage: python3 plot_acc_noise_distribution.py /path/to/roll_acc_only.csv")
        sys.exit(1)

    path = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/data/roll_acc_only1.csv"
    df = load_csv_clean(path)

    # Skip startup transient if desired
    t0 = float(df["time_s"].min())
    df = df[df["time_s"] >= (t0 + SKIP_FIRST_S)].reset_index(drop=True)

    # Plot roll vs time with ground truth
    plot_roll_vs_time(df)

    # Noise = measurement - ground truth (0 deg)
    noise = df["roll_acc_deg"].to_numpy()

    # Optional outlier clipping (useful if you bumped the sensor once)
    if OUTLIER_CLIP_STD is not None and noise.size > 20:
        mu0 = float(np.mean(noise))
        sigma0 = float(np.std(noise, ddof=1))
        if sigma0 > 1e-12:
            keep = np.abs(noise - mu0) <= OUTLIER_CLIP_STD * sigma0
            noise = noise[keep]

    mu = float(np.mean(noise))
    sigma = float(np.std(noise, ddof=1)) if noise.size > 1 else 0.0

    print("\n=== Accelerometer Roll Noise Stats (GT = 0°) ===")
    print(f"Samples used : {noise.size}")
    print(f"Mean (bias)  : {mu:+.6f} deg")
    print(f"Std dev      : {sigma:.6f} deg")
    print(f"Variance     : {sigma*sigma:.6f} deg^2")

    plot_noise_distribution(noise, mu=mu, sigma=sigma)


if __name__ == "__main__":
    main()



=== Accelerometer Roll Noise Stats (GT = 0°) ===
Samples used : 400
Mean (bias)  : -0.639879 deg
Std dev      : 0.026686 deg
Variance     : 0.000712 deg^2


In [55]:
# plot_roll_acc_motion.py
# -----------------------------------------------------------------------------
# Plot accelerometer-only roll data (no ground truth) for a motion experiment.
#
# Input CSV format (from your ESP32 accel logger):
#   time_s, dt_s, ax_g, ay_g, az_g, roll_acc_deg
#
# Usage:
#   python3 plot_roll_acc_motion.py /path/to/roll_acc_only.csv
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as go

TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]


def load_csv_clean(path: str) -> pd.DataFrame:
    """Loads CSV and ignores miniterm banners / dump markers if present."""
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    cleaned = []
    for ln in lines:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):  # miniterm banners / CSV DUMP markers
            continue
        cleaned.append(ln)

    df = pd.read_csv(StringIO("".join(cleaned)))

    expected = ["time_s", "dt_s", "ax_g", "ay_g", "az_g", "roll_acc_deg"]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}\nFound: {list(df.columns)}")

    for c in expected:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=expected).sort_values("time_s").reset_index(drop=True)
    return df


def plot_roll(df: pd.DataFrame):
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    fig.add_trace(go.Scatter(
        x=df["time_s"],
        y=df["roll_acc_deg"],
        mode="lines",
        name="roll_acc_deg",
        line=dict(width=2),
    ))

    fig.update_layout(
        title="Accelerometer-only Roll vs Time (Motion)",
        xaxis_title="time (s)",
        yaxis_title="roll (deg)",
        hovermode="x unified",
        legend_title="signal",
    )
    fig.show()


def plot_acc_axes(df: pd.DataFrame):
    """Optional: plot ax, ay, az to see when linear acceleration affects roll."""
    fig = go.Figure()
    fig.update_layout(template=TEMPLATE, colorway=COLORWAY)

    for col in ["ax_g", "ay_g", "az_g"]:
        fig.add_trace(go.Scatter(
            x=df["time_s"],
            y=df[col],
            mode="lines",
            name=col,
            line=dict(width=2),
        ))

    fig.update_layout(
        title="Accelerometer Axes vs Time",
        xaxis_title="time (s)",
        yaxis_title="acceleration (g)",
        hovermode="x unified",
        legend_title="axes",
    )
    fig.show()


def main():
    if len(sys.argv) < 2:
        print("Usage: python3 plot_roll_acc_motion.py /path/to/roll_acc_only.csv")
        sys.exit(1)

    path = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/data/roll_acc_only1.csv"
    df = load_csv_clean(path)

    plot_roll(df)
    plot_acc_axes(df)  # comment this out if you only want roll


if __name__ == "__main__":
    main()


In [56]:
#!/usr/bin/env python3
# plot_kf_partc_report_plots.py
# -----------------------------------------------------------------------------
# SC651 Assignment — Part (c)/(d): Kalman Filter (gyro + accel) plotting script
#
# Jupyter-safe + CLI-safe version.
#
# Usage (terminal):
#   python3 plot_kf_partc_report_plots.py /path/to/roll_kf_partc.csv
#
# Usage (Jupyter):
#   - Just run the cell (it will use DEFAULT_PATH), OR
#   - Set DEFAULT_PATH below to your file
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: skip first few seconds (startup transients)
SKIP_FIRST_S = 1.0

# Optional: clip outliers for residual histogram (if you bumped the sensor)
OUTLIER_CLIP_STD = 6.0

# -----------------------------
# SET THIS (JUPYTER FRIENDLY)
# -----------------------------
DEFAULT_PATH = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/data/roll_kf_partc.csv"


# -----------------------------------------------------------------------------
# CSV loading / cleaning
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Loads the Part (c) CSV and ignores:
      - miniterm banners
      - '--- CSV DUMP START ---' / '--- CSV DUMP END ---'
      - any other lines starting with '---'
    Robustly starts from the real header line: 'time_s,Ts_s,...'
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    cleaned = []
    header_found = False

    for ln in lines:
        s = ln.strip()
        if not s:
            continue

        # Skip banners / dump markers
        if s.startswith("---"):
            continue

        # Find the real CSV header line
        if not header_found:
            if s.startswith("time_s,"):
                header_found = True
                cleaned.append(ln)
            else:
                continue
        else:
            cleaned.append(ln)

    if not header_found:
        raise ValueError(
            "Could not find CSV header line starting with 'time_s,'. "
            "Check that you dumped the CSV correctly."
        )

    df = pd.read_csv(StringIO("".join(cleaned)))
    df.columns = [c.strip() for c in df.columns]

    expected = [
        "time_s", "Ts_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "ax_g", "ay_g", "az_g",
        "omega_roll_deg_s", "omega_used_deg_s",
        "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg",
        "theta_pred_deg", "residual_deg",
        "K", "P_pred", "P",
    ]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing columns: {missing}\nFound: {list(df.columns)}\n\n"
            "Tip: ensure your ESP32 sketch prints the header exactly as expected."
        )

    # Convert to numeric and drop broken rows
    for c in expected:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["time_s"]).sort_values("time_s").reset_index(drop=True)
    return df


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------
def _base_layout(fig: go.Figure, title: str, xlab: str, ylab: str) -> go.Figure:
    fig.update_layout(
        template=TEMPLATE,
        colorway=COLORWAY,
        title=title,
        xaxis_title=xlab,
        yaxis_title=ylab,
        hovermode="x unified",
        legend_title="signals",
    )
    return fig


def plot_roll_estimates(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"], mode="lines",
        name="roll_gyro_deg (integrated)", line=dict(width=2)
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (accelerometer)", line=dict(width=2)
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_kf_deg"], mode="lines",
        name="roll_kf_deg (Kalman)", line=dict(width=2)
    ))
    _base_layout(fig, "Part (c): Roll Estimates vs Time", "time (s)", "roll (deg)")
    fig.show()


def plot_gyro_rates(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_roll_deg_s"], mode="lines",
        name="omega_roll_deg_s (raw)", line=dict(width=2)
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_used_deg_s"], mode="lines",
        name="omega_used_deg_s (bias-corrected)", line=dict(width=2)
    ))
    _base_layout(fig, "Gyro Roll-Rate (raw vs bias-corrected)", "time (s)", "deg/s")
    fig.show()


def plot_acc_components(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ax_g"], mode="lines", name="ax_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ay_g"], mode="lines", name="ay_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["az_g"], mode="lines", name="az_g", line=dict(width=2)))
    _base_layout(fig, "Accelerometer Components vs Time", "time (s)", "acceleration (g)")
    fig.show()


def plot_residual_vs_time(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["residual_deg"], mode="lines",
        name="residual_deg = (theta_acc - theta_pred)", line=dict(width=2)
    ))
    fig.add_hline(y=0.0, line_dash="dash", opacity=0.6)
    _base_layout(fig, "Innovation / Residual vs Time", "time (s)", "residual (deg)")
    fig.show()


def plot_residual_distribution(df: pd.DataFrame) -> None:
    r = df["residual_deg"].dropna().to_numpy()

    if r.size == 0:
        print("WARNING: residual_deg empty; skipping residual distribution.")
        return

    # Optional outlier clipping for a clean histogram
    if OUTLIER_CLIP_STD is not None and r.size > 20:
        mu0 = float(np.mean(r))
        sig0 = float(np.std(r, ddof=1))
        if sig0 > 1e-12:
            keep = np.abs(r - mu0) <= OUTLIER_CLIP_STD * sig0
            r = r[keep]

    mu = float(np.mean(r))
    sig = float(np.std(r, ddof=1)) if r.size > 1 else 0.0
    med = float(np.median(r))

    df_r = pd.DataFrame({"residual_deg": r})

    fig = px.histogram(
        df_r,
        x="residual_deg",
        nbins=60,
        template=TEMPLATE,
        title="Residual Distribution (Innovation) — relates to measurement noise (σv)",
        color_discrete_sequence=[COLORWAY[0]],
    )
    fig.update_layout(xaxis_title="residual (deg)", yaxis_title="count")
    fig.add_vline(
        x=mu, line_width=2, line_dash="dash",
        annotation_text=f"mean={mu:.3f}°", annotation_position="top right"
    )
    fig.add_vline(x=mu + sig, line_width=1, line_dash="dot")
    fig.add_vline(x=mu - sig, line_width=1, line_dash="dot")
    fig.show()

    fig2 = px.box(
        df_r, y="residual_deg", template=TEMPLATE,
        title=f"Residual Spread (Box Plot) — median={med:.3f}°, std={sig:.3f}°"
    )
    fig2.update_layout(yaxis_title="residual (deg)")
    fig2.show()

    print("\n=== Residual Stats (useful for tuning notes) ===")
    print(f"Samples used : {r.size}")
    print(f"Mean         : {mu:+.6f} deg")
    print(f"Median       : {med:+.6f} deg")
    print(f"Std dev      : {sig:.6f} deg")
    print(f"Variance     : {sig*sig:.6f} deg^2")


def plot_gain_and_cov(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["K"], mode="lines", name="K (Kalman gain)", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["P_pred"], mode="lines", name="P_pred", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["P"], mode="lines", name="P (updated)", line=dict(width=2)))
    _base_layout(fig, "Kalman Gain and Covariance Evolution (tuning view)", "time (s)", "value")
    fig.show()


def plot_ts(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["Ts_s"], mode="lines", name="Ts_s", line=dict(width=2)))
    _base_layout(fig, "Actual Sample Time Ts (dt) vs Time", "time (s)", "Ts (s)")
    fig.show()


# -----------------------------------------------------------------------------
# Main (Jupyter-safe)
# -----------------------------------------------------------------------------
def _get_csv_path() -> Path:
    """
    In Jupyter, sys.argv may include kernel args like: ['...ipykernel_launcher.py','-f','...json'].
    So:
      - accept argv[1] only if it ends with '.csv'
      - else fall back to DEFAULT_PATH
    """
    if len(sys.argv) >= 2 and str(sys.argv[1]).lower().endswith(".csv"):
        return Path(sys.argv[1]).expanduser().resolve()
    return Path(DEFAULT_PATH).expanduser().resolve()


def main() -> None:
    csv_path = _get_csv_path()

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\n"
            f"Fix DEFAULT_PATH or pass a .csv path as argv[1]."
        )

    df = load_csv_clean(str(csv_path))

    # Skip startup transient (optional)
    t0 = float(df["time_s"].min())
    df = df[df["time_s"] >= (t0 + SKIP_FIRST_S)].reset_index(drop=True)

    # Quick Ts summary (part d mentions effect of Ts)
    ts = df["Ts_s"].dropna().to_numpy()
    if ts.size > 0:
        print("\n=== Ts (dt) Summary ===")
        print(f"Samples : {ts.size}")
        print(f"Mean    : {float(np.mean(ts)):.6f} s")
        print(f"Std     : {float(np.std(ts, ddof=1)):.6f} s" if ts.size > 1 else "Std     : 0.000000 s")
        print(f"Min/Max : {float(np.min(ts)):.6f} / {float(np.max(ts)):.6f} s")

    # Plots
    plot_roll_estimates(df)
    #plot_gyro_rates(df)
    #plot_acc_components(df)
    #plot_residual_vs_time(df)
    #plot_residual_distribution(df)
    plot_gain_and_cov(df)
    plot_ts(df)

    print("\nDone. (All figures were shown interactively.)")


if __name__ == "__main__":
    main()



=== Ts (dt) Summary ===
Samples : 3848
Mean    : 0.015331 s
Std     : 0.019815 s
Min/Max : 0.010957 / 0.135036 s



Done. (All figures were shown interactively.)


In [57]:
#!/usr/bin/env python3
# plot_kf_partc_report_plots.py
# -----------------------------------------------------------------------------
# SC651 Assignment — Part (c)/(d): Kalman Filter (gyro + accel) plotting script
#
# Robust to:
#   - SPIFFS dump markers, serial banners
#   - ---DONE--- line (serial streaming mode)
#   - corrupted/partial rows (common when serial stalls)
# Jupyter-safe + CLI-safe.
#
# CLI:
#   python3 plot_kf_partc_report_plots.py /path/to/file.csv
#
# Jupyter:
#   - Set DEFAULT_PATH below and run the cell, OR
#   - pass argv-like path by setting sys.argv manually (optional)
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: skip first few seconds (startup transients)
SKIP_FIRST_S = 1.0

# Optional: clip outliers for residual histogram
OUTLIER_CLIP_STD = 6.0

# -----------------------------
# SET THIS (JUPYTER FRIENDLY)
# -----------------------------
DEFAULT_PATH = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/run_one_twenty.csv"


# -----------------------------------------------------------------------------
# CSV loading / cleaning
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Robust CSV loader for KF logs.

    Handles:
      - miniterm banners, dump markers
      - ---DONE--- line
      - random corrupted rows (wrong column count)
      - extra whitespace

    Strategy:
      1) Find header line starting with "time_s,"
      2) Keep only lines with the same number of comma-separated fields as header
      3) pandas read_csv on the cleaned content
      4) numeric conversion + drop rows missing time_s
    """
    p = Path(path).expanduser().resolve()
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        raw_lines = f.readlines()

    header_idx = None
    header = None
    for i, ln in enumerate(raw_lines):
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            continue
        if s.lower().startswith("time_s,"):
            header_idx = i
            header = s
            break

    if header_idx is None or header is None:
        raise ValueError(
            "Could not find CSV header line starting with 'time_s,'. "
            "Make sure your file contains the header."
        )

    n_fields = header.count(",") + 1

    cleaned = [raw_lines[header_idx].strip() + "\n"]
    for ln in raw_lines[header_idx + 1 :]:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            # skip banners, dump markers, and ---DONE---
            continue

        # Fast structural check: correct number of fields
        if (s.count(",") + 1) != n_fields:
            continue

        cleaned.append(s + "\n")

    df = pd.read_csv(StringIO("".join(cleaned)))
    df.columns = [c.strip() for c in df.columns]

    # Expected columns for the FULL logger
    expected_full = [
        "time_s", "Ts_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "ax_g", "ay_g", "az_g",
        "omega_roll_deg_s", "omega_used_deg_s",
        "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg",
        "theta_pred_deg", "residual_deg",
        "K", "P_pred", "P",
    ]

    # We allow missing columns (in case you later log a minimal set).
    # But we REQUIRE at least these:
    required_min = ["time_s", "Ts_s", "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg"]

    missing_min = [c for c in required_min if c not in df.columns]
    if missing_min:
        raise ValueError(
            f"Missing required columns: {missing_min}\nFound: {list(df.columns)}"
        )

    # Convert whatever columns exist to numeric
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop broken rows
    df = df.dropna(subset=["time_s"]).sort_values("time_s").reset_index(drop=True)

    # If residual isn't present but theta_pred is, compute it
    if "residual_deg" not in df.columns and "theta_pred_deg" in df.columns:
        df["residual_deg"] = df["roll_acc_deg"] - df["theta_pred_deg"]

    return df


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------
def _base_layout(fig: go.Figure, title: str, xlab: str, ylab: str) -> go.Figure:
    fig.update_layout(
        template=TEMPLATE,
        colorway=COLORWAY,
        title=title,
        xaxis_title=xlab,
        yaxis_title=ylab,
        hovermode="x unified",
        legend_title="signals",
    )
    return fig


def plot_roll_estimates(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"], mode="lines",
        name="roll_gyro_deg (integrated)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (accelerometer)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_kf_deg"], mode="lines",
        name="roll_kf_deg (Kalman)", line=dict(width=2),
    ))
    _base_layout(fig, "Roll Estimates vs Time", "time (s)", "roll (deg)")
    fig.show()


def plot_gyro_rates(df: pd.DataFrame) -> None:
    if "omega_roll_deg_s" not in df.columns or "omega_used_deg_s" not in df.columns:
        print("[skip] gyro rates (omega_* columns not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_roll_deg_s"], mode="lines",
        name="omega_roll_deg_s (raw)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_used_deg_s"], mode="lines",
        name="omega_used_deg_s (bias-corrected)", line=dict(width=2),
    ))
    _base_layout(fig, "Gyro Roll-Rate (raw vs bias-corrected)", "time (s)", "deg/s")
    fig.show()


def plot_acc_components(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["ax_g", "ay_g", "az_g"]):
        print("[skip] accel components (ax_g/ay_g/az_g not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ax_g"], mode="lines", name="ax_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ay_g"], mode="lines", name="ay_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["az_g"], mode="lines", name="az_g", line=dict(width=2)))
    _base_layout(fig, "Accelerometer Components vs Time", "time (s)", "acceleration (g)")
    fig.show()


def plot_residual_vs_time(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual vs time (residual_deg not found and cannot be computed)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["residual_deg"], mode="lines",
        name="residual_deg = (theta_acc - theta_pred)", line=dict(width=2),
    ))
    fig.add_hline(y=0.0, line_dash="dash", opacity=0.6)
    _base_layout(fig, "Innovation / Residual vs Time", "time (s)", "residual (deg)")
    fig.show()


def plot_residual_distribution(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual distribution (residual_deg not found)")
        return

    r = df["residual_deg"].dropna().to_numpy()
    if r.size == 0:
        print("[skip] residual distribution (empty)")
        return

    if OUTLIER_CLIP_STD is not None and r.size > 20:
        mu0 = float(np.mean(r))
        sig0 = float(np.std(r, ddof=1))
        if sig0 > 1e-12:
            keep = np.abs(r - mu0) <= OUTLIER_CLIP_STD * sig0
            r = r[keep]

    mu = float(np.mean(r))
    sig = float(np.std(r, ddof=1)) if r.size > 1 else 0.0
    med = float(np.median(r))

    df_r = pd.DataFrame({"residual_deg": r})

    fig = px.histogram(
        df_r, x="residual_deg", nbins=60, template=TEMPLATE,
        title="Residual Distribution (Innovation)",
        color_discrete_sequence=[COLORWAY[0]],
    )
    fig.update_layout(xaxis_title="residual (deg)", yaxis_title="count")
    fig.add_vline(x=mu, line_width=2, line_dash="dash",
                  annotation_text=f"mean={mu:.3f}°", annotation_position="top right")
    fig.add_vline(x=mu + sig, line_width=1, line_dash="dot")
    fig.add_vline(x=mu - sig, line_width=1, line_dash="dot")
    fig.show()

    fig2 = px.box(
        df_r, y="residual_deg", template=TEMPLATE,
        title=f"Residual Spread — median={med:.3f}°, std={sig:.3f}°",
    )
    fig2.update_layout(yaxis_title="residual (deg)")
    fig2.show()

    print("\n=== Residual Stats ===")
    print(f"Samples used : {r.size}")
    print(f"Mean         : {mu:+.6f} deg")
    print(f"Median       : {med:+.6f} deg")
    print(f"Std dev      : {sig:.6f} deg")
    print(f"Variance     : {sig*sig:.6f} deg^2")


def plot_gain_and_cov(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["K", "P"]):
        print("[skip] gain/cov (K or P not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["K"], mode="lines", name="K (Kalman gain)", line=dict(width=2)))
    if "P_pred" in df.columns:
        fig.add_trace(go.Scatter(x=df["time_s"], y=df["P_pred"], mode="lines", name="P_pred", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["P"], mode="lines", name="P (updated)", line=dict(width=2)))
    _base_layout(fig, "Kalman Gain and Covariance Evolution", "time (s)", "value")
    fig.show()
    
def plot_pred_vs_meas(df: pd.DataFrame) -> None:
    # Needs theta_pred_deg and roll_acc_deg
    if "theta_pred_deg" not in df.columns:
        print("[skip] pred vs meas (theta_pred_deg not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["theta_pred_deg"], mode="lines",
        name="theta_pred_deg (prediction)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (measurement)", line=dict(width=2),
    ))
    _base_layout(fig, "Prediction vs Measurement (tuning diagnostic)", "time (s)", "deg")
    fig.show()



def plot_ts(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["Ts_s"], mode="lines", name="Ts_s", line=dict(width=2)))
    _base_layout(fig, "Actual Sample Time Ts (dt) vs Time", "time (s)", "Ts (s)")
    fig.show()


# -----------------------------------------------------------------------------
# Main (Jupyter-safe)
# -----------------------------------------------------------------------------
def _get_csv_path() -> Path:
    """
    In Jupyter, sys.argv may contain kernel flags.
    So:
      - accept argv[1] only if it ends with '.csv'
      - else fall back to DEFAULT_PATH
    """
    if len(sys.argv) >= 2 and str(sys.argv[1]).lower().endswith(".csv"):
        return Path(sys.argv[1]).expanduser().resolve()
    return Path(DEFAULT_PATH).expanduser().resolve()


def main() -> None:
    csv_path = _get_csv_path()

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\nFix DEFAULT_PATH or pass a .csv path as argv[1]."
        )

    df = load_csv_clean(str(csv_path))

    # Skip startup transient
    if SKIP_FIRST_S and SKIP_FIRST_S > 0:
        t0 = float(df["time_s"].min())
        df = df[df["time_s"] >= (t0 + SKIP_FIRST_S)].reset_index(drop=True)

    # Ts summary
    ts = df["Ts_s"].dropna().to_numpy()
    if ts.size > 0:
        print("\n=== Ts (dt) Summary ===")
        print(f"Samples : {ts.size}")
        print(f"Mean    : {float(np.mean(ts)):.6f} s")
        if ts.size > 1:
            print(f"Std     : {float(np.std(ts, ddof=1)):.6f} s")
        print(f"Min/Max : {float(np.min(ts)):.6f} / {float(np.max(ts)):.6f} s")

    # Choose what plots you want (no more commenting/uncommenting)
    plots_to_show = [
        "roll",
        "pred_meas",
        "gain",
        "ts",
        # "gyro",
        # "acc",
        # "residual_time",
        # "residual_hist",
    ]

    if "roll" in plots_to_show:
        plot_roll_estimates(df)
    if "gyro" in plots_to_show:
        plot_gyro_rates(df)
    if "acc" in plots_to_show:
        plot_acc_components(df)
    if "residual_time" in plots_to_show:
        plot_residual_vs_time(df)
    if "residual_hist" in plots_to_show:
        plot_residual_distribution(df)
    if "gain" in plots_to_show:
        plot_gain_and_cov(df)
    if "ts" in plots_to_show:
        plot_ts(df)
    if "pred_meas" in plots_to_show:
        plot_pred_vs_meas(df)


    print("\nDone. (All figures were shown interactively.)")


if __name__ == "__main__":
    main()



=== Ts (dt) Summary ===
Samples : 1134
Mean    : 0.021000 s
Std     : 0.000002 s
Min/Max : 0.020988 / 0.021012 s



Done. (All figures were shown interactively.)


In [58]:
#!/usr/bin/env python3
# plot_kf_partc_report_plots.py
# -----------------------------------------------------------------------------
# SC651 Assignment — Part (c)/(d): Kalman Filter (gyro + accel) plotting script
#
# Robust to:
#   - SPIFFS dump markers, serial banners
#   - ---DONE--- line (serial streaming mode)
#   - corrupted/partial rows (common when serial stalls)
# Jupyter-safe + CLI-safe.
#
# CLI:
#   python3 plot_kf_partc_report_plots.py /path/to/file.csv
#
# Jupyter:
#   - Set DEFAULT_PATH below and run the cell, OR
#   - pass argv-like path by setting sys.argv manually (optional)
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: skip first few seconds (startup transients)
SKIP_FIRST_S = 1.0

# Optional: clip outliers for residual histogram
OUTLIER_CLIP_STD = 6.0

# -----------------------------
# SET THIS (JUPYTER FRIENDLY)
# -----------------------------
DEFAULT_PATH = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/run_one_hundred.csv"


# -----------------------------------------------------------------------------
# CSV loading / cleaning
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Robust CSV loader for KF logs.

    Handles:
      - miniterm banners, dump markers
      - ---DONE--- line
      - random corrupted rows (wrong column count)
      - extra whitespace

    Strategy:
      1) Find header line starting with "time_s,"
      2) Keep only lines with the same number of comma-separated fields as header
      3) pandas read_csv on the cleaned content
      4) numeric conversion + drop rows missing time_s
    """
    p = Path(path).expanduser().resolve()
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        raw_lines = f.readlines()

    header_idx = None
    header = None
    for i, ln in enumerate(raw_lines):
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            continue
        if s.lower().startswith("time_s,"):
            header_idx = i
            header = s
            break

    if header_idx is None or header is None:
        raise ValueError(
            "Could not find CSV header line starting with 'time_s,'. "
            "Make sure your file contains the header."
        )

    n_fields = header.count(",") + 1

    cleaned = [raw_lines[header_idx].strip() + "\n"]
    for ln in raw_lines[header_idx + 1 :]:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            # skip banners, dump markers, and ---DONE---
            continue

        # Fast structural check: correct number of fields
        if (s.count(",") + 1) != n_fields:
            continue

        cleaned.append(s + "\n")

    df = pd.read_csv(StringIO("".join(cleaned)))
    df.columns = [c.strip() for c in df.columns]

    # Expected columns for the FULL logger
    expected_full = [
        "time_s", "Ts_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "ax_g", "ay_g", "az_g",
        "omega_roll_deg_s", "omega_used_deg_s",
        "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg",
        "theta_pred_deg", "residual_deg",
        "K", "P_pred", "P",
    ]

    # We allow missing columns (in case you later log a minimal set).
    # But we REQUIRE at least these:
    required_min = ["time_s", "Ts_s", "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg"]

    missing_min = [c for c in required_min if c not in df.columns]
    if missing_min:
        raise ValueError(
            f"Missing required columns: {missing_min}\nFound: {list(df.columns)}"
        )

    # Convert whatever columns exist to numeric
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop broken rows
    df = df.dropna(subset=["time_s"]).sort_values("time_s").reset_index(drop=True)

    # If residual isn't present but theta_pred is, compute it
    if "residual_deg" not in df.columns and "theta_pred_deg" in df.columns:
        df["residual_deg"] = df["roll_acc_deg"] - df["theta_pred_deg"]

    return df


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------
def _base_layout(fig: go.Figure, title: str, xlab: str, ylab: str) -> go.Figure:
    fig.update_layout(
        template=TEMPLATE,
        colorway=COLORWAY,
        title=title,
        xaxis_title=xlab,
        yaxis_title=ylab,
        hovermode="x unified",
        legend_title="signals",
    )
    return fig


def plot_roll_estimates(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"], mode="lines",
        name="roll_gyro_deg (integrated)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (accelerometer)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_kf_deg"], mode="lines",
        name="roll_kf_deg (Kalman)", line=dict(width=2),
    ))
    _base_layout(fig, "Roll Estimates vs Time", "time (s)", "roll (deg)")
    fig.show()


def plot_gyro_rates(df: pd.DataFrame) -> None:
    if "omega_roll_deg_s" not in df.columns or "omega_used_deg_s" not in df.columns:
        print("[skip] gyro rates (omega_* columns not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_roll_deg_s"], mode="lines",
        name="omega_roll_deg_s (raw)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_used_deg_s"], mode="lines",
        name="omega_used_deg_s (bias-corrected)", line=dict(width=2),
    ))
    _base_layout(fig, "Gyro Roll-Rate (raw vs bias-corrected)", "time (s)", "deg/s")
    fig.show()


def plot_acc_components(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["ax_g", "ay_g", "az_g"]):
        print("[skip] accel components (ax_g/ay_g/az_g not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ax_g"], mode="lines", name="ax_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ay_g"], mode="lines", name="ay_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["az_g"], mode="lines", name="az_g", line=dict(width=2)))
    _base_layout(fig, "Accelerometer Components vs Time", "time (s)", "acceleration (g)")
    fig.show()


def plot_residual_vs_time(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual vs time (residual_deg not found and cannot be computed)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["residual_deg"], mode="lines",
        name="residual_deg = (theta_acc - theta_pred)", line=dict(width=2),
    ))
    fig.add_hline(y=0.0, line_dash="dash", opacity=0.6)
    _base_layout(fig, "Innovation / Residual vs Time", "time (s)", "residual (deg)")
    fig.show()


def plot_residual_distribution(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual distribution (residual_deg not found)")
        return

    r = df["residual_deg"].dropna().to_numpy()
    if r.size == 0:
        print("[skip] residual distribution (empty)")
        return

    if OUTLIER_CLIP_STD is not None and r.size > 20:
        mu0 = float(np.mean(r))
        sig0 = float(np.std(r, ddof=1))
        if sig0 > 1e-12:
            keep = np.abs(r - mu0) <= OUTLIER_CLIP_STD * sig0
            r = r[keep]

    mu = float(np.mean(r))
    sig = float(np.std(r, ddof=1)) if r.size > 1 else 0.0
    med = float(np.median(r))

    df_r = pd.DataFrame({"residual_deg": r})

    fig = px.histogram(
        df_r, x="residual_deg", nbins=60, template=TEMPLATE,
        title="Residual Distribution (Innovation)",
        color_discrete_sequence=[COLORWAY[0]],
    )
    fig.update_layout(xaxis_title="residual (deg)", yaxis_title="count")
    fig.add_vline(x=mu, line_width=2, line_dash="dash",
                  annotation_text=f"mean={mu:.3f}°", annotation_position="top right")
    fig.add_vline(x=mu + sig, line_width=1, line_dash="dot")
    fig.add_vline(x=mu - sig, line_width=1, line_dash="dot")
    fig.show()

    fig2 = px.box(
        df_r, y="residual_deg", template=TEMPLATE,
        title=f"Residual Spread — median={med:.3f}°, std={sig:.3f}°",
    )
    fig2.update_layout(yaxis_title="residual (deg)")
    fig2.show()

    print("\n=== Residual Stats ===")
    print(f"Samples used : {r.size}")
    print(f"Mean         : {mu:+.6f} deg")
    print(f"Median       : {med:+.6f} deg")
    print(f"Std dev      : {sig:.6f} deg")
    print(f"Variance     : {sig*sig:.6f} deg^2")


def plot_gain_and_cov(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["K", "P"]):
        print("[skip] gain/cov (K or P not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["K"], mode="lines", name="K (Kalman gain)", line=dict(width=2)))
    if "P_pred" in df.columns:
        fig.add_trace(go.Scatter(x=df["time_s"], y=df["P_pred"], mode="lines", name="P_pred", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["P"], mode="lines", name="P (updated)", line=dict(width=2)))
    _base_layout(fig, "Kalman Gain and Covariance Evolution", "time (s)", "value")
    fig.show()
    
def plot_pred_vs_meas(df: pd.DataFrame) -> None:
    # Needs theta_pred_deg and roll_acc_deg
    if "theta_pred_deg" not in df.columns:
        print("[skip] pred vs meas (theta_pred_deg not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["theta_pred_deg"], mode="lines",
        name="theta_pred_deg (prediction)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (measurement)", line=dict(width=2),
    ))
    _base_layout(fig, "Prediction vs Measurement (tuning diagnostic)", "time (s)", "deg")
    fig.show()



def plot_ts(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["Ts_s"], mode="lines", name="Ts_s", line=dict(width=2)))
    _base_layout(fig, "Actual Sample Time Ts (dt) vs Time", "time (s)", "Ts (s)")
    fig.show()


# -----------------------------------------------------------------------------
# Main (Jupyter-safe)
# -----------------------------------------------------------------------------
def _get_csv_path() -> Path:
    """
    In Jupyter, sys.argv may contain kernel flags.
    So:
      - accept argv[1] only if it ends with '.csv'
      - else fall back to DEFAULT_PATH
    """
    if len(sys.argv) >= 2 and str(sys.argv[1]).lower().endswith(".csv"):
        return Path(sys.argv[1]).expanduser().resolve()
    return Path(DEFAULT_PATH).expanduser().resolve()


def main() -> None:
    csv_path = _get_csv_path()

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\nFix DEFAULT_PATH or pass a .csv path as argv[1]."
        )

    df = load_csv_clean(str(csv_path))

    # Skip startup transient
    if SKIP_FIRST_S and SKIP_FIRST_S > 0:
        t0 = float(df["time_s"].min())
        df = df[df["time_s"] >= (t0 + SKIP_FIRST_S)].reset_index(drop=True)

    # Ts summary
    ts = df["Ts_s"].dropna().to_numpy()
    if ts.size > 0:
        print("\n=== Ts (dt) Summary ===")
        print(f"Samples : {ts.size}")
        print(f"Mean    : {float(np.mean(ts)):.6f} s")
        if ts.size > 1:
            print(f"Std     : {float(np.std(ts, ddof=1)):.6f} s")
        print(f"Min/Max : {float(np.min(ts)):.6f} / {float(np.max(ts)):.6f} s")

    # Choose what plots you want (no more commenting/uncommenting)
    plots_to_show = [
        "roll",
        "pred_meas",
        "gain",
        "ts",
        # "gyro",
        # "acc",
        # "residual_time",
        # "residual_hist",
    ]

    if "roll" in plots_to_show:
        plot_roll_estimates(df)
    if "gyro" in plots_to_show:
        plot_gyro_rates(df)
    if "acc" in plots_to_show:
        plot_acc_components(df)
    if "residual_time" in plots_to_show:
        plot_residual_vs_time(df)
    if "residual_hist" in plots_to_show:
        plot_residual_distribution(df)
    if "gain" in plots_to_show:
        plot_gain_and_cov(df)
    if "ts" in plots_to_show:
        plot_ts(df)
    if "pred_meas" in plots_to_show:
        plot_pred_vs_meas(df)


    print("\nDone. (All figures were shown interactively.)")


if __name__ == "__main__":
    main()



=== Ts (dt) Summary ===
Samples : 1395
Mean    : 0.021000 s
Std     : 0.000001 s
Min/Max : 0.020988 / 0.021012 s



Done. (All figures were shown interactively.)


In [59]:
#!/usr/bin/env python3
# plot_kf_partc_report_plots.py
# -----------------------------------------------------------------------------
# SC651 Assignment — Part (c)/(d): Kalman Filter (gyro + accel) plotting script
#
# Robust to:
#   - SPIFFS dump markers, serial banners
#   - ---DONE--- line (serial streaming mode)
#   - corrupted/partial rows (common when serial stalls)
# Jupyter-safe + CLI-safe.
#
# CLI:
#   python3 plot_kf_partc_report_plots.py /path/to/file.csv
#
# Jupyter:
#   - Set DEFAULT_PATH below and run the cell, OR
#   - pass argv-like path by setting sys.argv manually (optional)
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: skip first few seconds (startup transients)
SKIP_FIRST_S = 1.0

# Optional: clip outliers for residual histogram
OUTLIER_CLIP_STD = 6.0

# -----------------------------
# SET THIS (JUPYTER FRIENDLY)
# -----------------------------
DEFAULT_PATH = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/run_one_thousand.csv"


# -----------------------------------------------------------------------------
# CSV loading / cleaning
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Robust CSV loader for KF logs.

    Handles:
      - miniterm banners, dump markers
      - ---DONE--- line
      - random corrupted rows (wrong column count)
      - extra whitespace

    Strategy:
      1) Find header line starting with "time_s,"
      2) Keep only lines with the same number of comma-separated fields as header
      3) pandas read_csv on the cleaned content
      4) numeric conversion + drop rows missing time_s
    """
    p = Path(path).expanduser().resolve()
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        raw_lines = f.readlines()

    header_idx = None
    header = None
    for i, ln in enumerate(raw_lines):
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            continue
        if s.lower().startswith("time_s,"):
            header_idx = i
            header = s
            break

    if header_idx is None or header is None:
        raise ValueError(
            "Could not find CSV header line starting with 'time_s,'. "
            "Make sure your file contains the header."
        )

    n_fields = header.count(",") + 1

    cleaned = [raw_lines[header_idx].strip() + "\n"]
    for ln in raw_lines[header_idx + 1 :]:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            # skip banners, dump markers, and ---DONE---
            continue

        # Fast structural check: correct number of fields
        if (s.count(",") + 1) != n_fields:
            continue

        cleaned.append(s + "\n")

    df = pd.read_csv(StringIO("".join(cleaned)))
    df.columns = [c.strip() for c in df.columns]

    # Expected columns for the FULL logger
    expected_full = [
        "time_s", "Ts_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "ax_g", "ay_g", "az_g",
        "omega_roll_deg_s", "omega_used_deg_s",
        "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg",
        "theta_pred_deg", "residual_deg",
        "K", "P_pred", "P",
    ]

    # We allow missing columns (in case you later log a minimal set).
    # But we REQUIRE at least these:
    required_min = ["time_s", "Ts_s", "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg"]

    missing_min = [c for c in required_min if c not in df.columns]
    if missing_min:
        raise ValueError(
            f"Missing required columns: {missing_min}\nFound: {list(df.columns)}"
        )

    # Convert whatever columns exist to numeric
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop broken rows
    df = df.dropna(subset=["time_s"]).sort_values("time_s").reset_index(drop=True)

    # If residual isn't present but theta_pred is, compute it
    if "residual_deg" not in df.columns and "theta_pred_deg" in df.columns:
        df["residual_deg"] = df["roll_acc_deg"] - df["theta_pred_deg"]

    return df


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------
def _base_layout(fig: go.Figure, title: str, xlab: str, ylab: str) -> go.Figure:
    fig.update_layout(
        template=TEMPLATE,
        colorway=COLORWAY,
        title=title,
        xaxis_title=xlab,
        yaxis_title=ylab,
        hovermode="x unified",
        legend_title="signals",
    )
    return fig


def plot_roll_estimates(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"], mode="lines",
        name="roll_gyro_deg (integrated)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (accelerometer)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_kf_deg"], mode="lines",
        name="roll_kf_deg (Kalman)", line=dict(width=2),
    ))
    _base_layout(fig, "Roll Estimates vs Time", "time (s)", "roll (deg)")
    fig.show()


def plot_gyro_rates(df: pd.DataFrame) -> None:
    if "omega_roll_deg_s" not in df.columns or "omega_used_deg_s" not in df.columns:
        print("[skip] gyro rates (omega_* columns not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_roll_deg_s"], mode="lines",
        name="omega_roll_deg_s (raw)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_used_deg_s"], mode="lines",
        name="omega_used_deg_s (bias-corrected)", line=dict(width=2),
    ))
    _base_layout(fig, "Gyro Roll-Rate (raw vs bias-corrected)", "time (s)", "deg/s")
    fig.show()


def plot_acc_components(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["ax_g", "ay_g", "az_g"]):
        print("[skip] accel components (ax_g/ay_g/az_g not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ax_g"], mode="lines", name="ax_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ay_g"], mode="lines", name="ay_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["az_g"], mode="lines", name="az_g", line=dict(width=2)))
    _base_layout(fig, "Accelerometer Components vs Time", "time (s)", "acceleration (g)")
    fig.show()


def plot_residual_vs_time(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual vs time (residual_deg not found and cannot be computed)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["residual_deg"], mode="lines",
        name="residual_deg = (theta_acc - theta_pred)", line=dict(width=2),
    ))
    fig.add_hline(y=0.0, line_dash="dash", opacity=0.6)
    _base_layout(fig, "Innovation / Residual vs Time", "time (s)", "residual (deg)")
    fig.show()


def plot_residual_distribution(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual distribution (residual_deg not found)")
        return

    r = df["residual_deg"].dropna().to_numpy()
    if r.size == 0:
        print("[skip] residual distribution (empty)")
        return

    if OUTLIER_CLIP_STD is not None and r.size > 20:
        mu0 = float(np.mean(r))
        sig0 = float(np.std(r, ddof=1))
        if sig0 > 1e-12:
            keep = np.abs(r - mu0) <= OUTLIER_CLIP_STD * sig0
            r = r[keep]

    mu = float(np.mean(r))
    sig = float(np.std(r, ddof=1)) if r.size > 1 else 0.0
    med = float(np.median(r))

    df_r = pd.DataFrame({"residual_deg": r})

    fig = px.histogram(
        df_r, x="residual_deg", nbins=60, template=TEMPLATE,
        title="Residual Distribution (Innovation)",
        color_discrete_sequence=[COLORWAY[0]],
    )
    fig.update_layout(xaxis_title="residual (deg)", yaxis_title="count")
    fig.add_vline(x=mu, line_width=2, line_dash="dash",
                  annotation_text=f"mean={mu:.3f}°", annotation_position="top right")
    fig.add_vline(x=mu + sig, line_width=1, line_dash="dot")
    fig.add_vline(x=mu - sig, line_width=1, line_dash="dot")
    fig.show()

    fig2 = px.box(
        df_r, y="residual_deg", template=TEMPLATE,
        title=f"Residual Spread — median={med:.3f}°, std={sig:.3f}°",
    )
    fig2.update_layout(yaxis_title="residual (deg)")
    fig2.show()

    print("\n=== Residual Stats ===")
    print(f"Samples used : {r.size}")
    print(f"Mean         : {mu:+.6f} deg")
    print(f"Median       : {med:+.6f} deg")
    print(f"Std dev      : {sig:.6f} deg")
    print(f"Variance     : {sig*sig:.6f} deg^2")


def plot_gain_and_cov(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["K", "P"]):
        print("[skip] gain/cov (K or P not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["K"], mode="lines", name="K (Kalman gain)", line=dict(width=2)))
    if "P_pred" in df.columns:
        fig.add_trace(go.Scatter(x=df["time_s"], y=df["P_pred"], mode="lines", name="P_pred", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["P"], mode="lines", name="P (updated)", line=dict(width=2)))
    _base_layout(fig, "Kalman Gain and Covariance Evolution", "time (s)", "value")
    fig.show()
    
def plot_pred_vs_meas(df: pd.DataFrame) -> None:
    # Needs theta_pred_deg and roll_acc_deg
    if "theta_pred_deg" not in df.columns:
        print("[skip] pred vs meas (theta_pred_deg not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["theta_pred_deg"], mode="lines",
        name="theta_pred_deg (prediction)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (measurement)", line=dict(width=2),
    ))
    _base_layout(fig, "Prediction vs Measurement (tuning diagnostic)", "time (s)", "deg")
    fig.show()



def plot_ts(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["Ts_s"], mode="lines", name="Ts_s", line=dict(width=2)))
    _base_layout(fig, "Actual Sample Time Ts (dt) vs Time", "time (s)", "Ts (s)")
    fig.show()


# -----------------------------------------------------------------------------
# Main (Jupyter-safe)
# -----------------------------------------------------------------------------
def _get_csv_path() -> Path:
    """
    In Jupyter, sys.argv may contain kernel flags.
    So:
      - accept argv[1] only if it ends with '.csv'
      - else fall back to DEFAULT_PATH
    """
    if len(sys.argv) >= 2 and str(sys.argv[1]).lower().endswith(".csv"):
        return Path(sys.argv[1]).expanduser().resolve()
    return Path(DEFAULT_PATH).expanduser().resolve()


def main() -> None:
    csv_path = _get_csv_path()

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\nFix DEFAULT_PATH or pass a .csv path as argv[1]."
        )

    df = load_csv_clean(str(csv_path))

    # Skip startup transient
    if SKIP_FIRST_S and SKIP_FIRST_S > 0:
        t0 = float(df["time_s"].min())
        df = df[df["time_s"] >= (t0 + SKIP_FIRST_S)].reset_index(drop=True)

    # Ts summary
    ts = df["Ts_s"].dropna().to_numpy()
    if ts.size > 0:
        print("\n=== Ts (dt) Summary ===")
        print(f"Samples : {ts.size}")
        print(f"Mean    : {float(np.mean(ts)):.6f} s")
        if ts.size > 1:
            print(f"Std     : {float(np.std(ts, ddof=1)):.6f} s")
        print(f"Min/Max : {float(np.min(ts)):.6f} / {float(np.max(ts)):.6f} s")

    # Choose what plots you want (no more commenting/uncommenting)
    plots_to_show = [
        "roll",
        "pred_meas",
        "gain",
        "ts",
        # "gyro",
        # "acc",
        # "residual_time",
        # "residual_hist",
    ]

    if "roll" in plots_to_show:
        plot_roll_estimates(df)
    if "gyro" in plots_to_show:
        plot_gyro_rates(df)
    if "acc" in plots_to_show:
        plot_acc_components(df)
    if "residual_time" in plots_to_show:
        plot_residual_vs_time(df)
    if "residual_hist" in plots_to_show:
        plot_residual_distribution(df)
    if "gain" in plots_to_show:
        plot_gain_and_cov(df)
    if "ts" in plots_to_show:
        plot_ts(df)
    if "pred_meas" in plots_to_show:
        plot_pred_vs_meas(df)


    print("\nDone. (All figures were shown interactively.)")


if __name__ == "__main__":
    main()



=== Ts (dt) Summary ===
Samples : 1889
Mean    : 0.021000 s
Std     : 0.000003 s
Min/Max : 0.020988 / 0.021012 s



Done. (All figures were shown interactively.)


In [3]:
#!/usr/bin/env python3
# plot_kf_partc_report_plots.py
# -----------------------------------------------------------------------------
# SC651 Assignment — Part (c)/(d): Kalman Filter (gyro + accel) plotting script
#
# Robust to:
#   - SPIFFS dump markers, serial banners
#   - ---DONE--- line (serial streaming mode)
#   - corrupted/partial rows (common when serial stalls)
# Jupyter-safe + CLI-safe.
#
# CLI:
#   python3 plot_kf_partc_report_plots.py /path/to/file.csv
#
# Jupyter:
#   - Set DEFAULT_PATH below and run the cell, OR
#   - pass argv-like path by setting sys.argv manually (optional)
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: skip first few seconds (startup transients)
SKIP_FIRST_S = 1.0

# Optional: clip outliers for residual histogram
OUTLIER_CLIP_STD = 6.0

# -----------------------------
# SET THIS (JUPYTER FRIENDLY)
# -----------------------------
DEFAULT_PATH = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/run_zero_point_zero_one.csv"


# -----------------------------------------------------------------------------
# CSV loading / cleaning
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Robust CSV loader for KF logs.

    Handles:
      - miniterm banners, dump markers
      - ---DONE--- line
      - random corrupted rows (wrong column count)
      - extra whitespace

    Strategy:
      1) Find header line starting with "time_s,"
      2) Keep only lines with the same number of comma-separated fields as header
      3) pandas read_csv on the cleaned content
      4) numeric conversion + drop rows missing time_s
    """
    p = Path(path).expanduser().resolve()
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        raw_lines = f.readlines()

    header_idx = None
    header = None
    for i, ln in enumerate(raw_lines):
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            continue
        if s.lower().startswith("time_s,"):
            header_idx = i
            header = s
            break

    if header_idx is None or header is None:
        raise ValueError(
            "Could not find CSV header line starting with 'time_s,'. "
            "Make sure your file contains the header."
        )

    n_fields = header.count(",") + 1

    cleaned = [raw_lines[header_idx].strip() + "\n"]
    for ln in raw_lines[header_idx + 1 :]:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            # skip banners, dump markers, and ---DONE---
            continue

        # Fast structural check: correct number of fields
        if (s.count(",") + 1) != n_fields:
            continue

        cleaned.append(s + "\n")

    df = pd.read_csv(StringIO("".join(cleaned)))
    df.columns = [c.strip() for c in df.columns]

    # Expected columns for the FULL logger
    expected_full = [
        "time_s", "Ts_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "ax_g", "ay_g", "az_g",
        "omega_roll_deg_s", "omega_used_deg_s",
        "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg",
        "theta_pred_deg", "residual_deg",
        "K", "P_pred", "P",
    ]

    # We allow missing columns (in case you later log a minimal set).
    # But we REQUIRE at least these:
    required_min = ["time_s", "Ts_s", "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg"]

    missing_min = [c for c in required_min if c not in df.columns]
    if missing_min:
        raise ValueError(
            f"Missing required columns: {missing_min}\nFound: {list(df.columns)}"
        )

    # Convert whatever columns exist to numeric
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop broken rows
    df = df.dropna(subset=["time_s"]).sort_values("time_s").reset_index(drop=True)

    # If residual isn't present but theta_pred is, compute it
    if "residual_deg" not in df.columns and "theta_pred_deg" in df.columns:
        df["residual_deg"] = df["roll_acc_deg"] - df["theta_pred_deg"]

    return df


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------
def _base_layout(fig: go.Figure, title: str, xlab: str, ylab: str) -> go.Figure:
    fig.update_layout(
        template=TEMPLATE,
        colorway=COLORWAY,
        title=title,
        xaxis_title=xlab,
        yaxis_title=ylab,
        hovermode="x unified",
        legend_title="signals",
    )
    return fig


def plot_roll_estimates(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"], mode="lines",
        name="roll_gyro_deg (integrated)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (accelerometer)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_kf_deg"], mode="lines",
        name="roll_kf_deg (Kalman)", line=dict(width=2),
    ))
    _base_layout(fig, "Roll Estimates vs Time", "time (s)", "roll (deg)")
    fig.show()


def plot_gyro_rates(df: pd.DataFrame) -> None:
    if "omega_roll_deg_s" not in df.columns or "omega_used_deg_s" not in df.columns:
        print("[skip] gyro rates (omega_* columns not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_roll_deg_s"], mode="lines",
        name="omega_roll_deg_s (raw)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_used_deg_s"], mode="lines",
        name="omega_used_deg_s (bias-corrected)", line=dict(width=2),
    ))
    _base_layout(fig, "Gyro Roll-Rate (raw vs bias-corrected)", "time (s)", "deg/s")
    fig.show()


def plot_acc_components(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["ax_g", "ay_g", "az_g"]):
        print("[skip] accel components (ax_g/ay_g/az_g not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ax_g"], mode="lines", name="ax_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ay_g"], mode="lines", name="ay_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["az_g"], mode="lines", name="az_g", line=dict(width=2)))
    _base_layout(fig, "Accelerometer Components vs Time", "time (s)", "acceleration (g)")
    fig.show()


def plot_residual_vs_time(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual vs time (residual_deg not found and cannot be computed)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["residual_deg"], mode="lines",
        name="residual_deg = (theta_acc - theta_pred)", line=dict(width=2),
    ))
    fig.add_hline(y=0.0, line_dash="dash", opacity=0.6)
    _base_layout(fig, "Innovation / Residual vs Time", "time (s)", "residual (deg)")
    fig.show()


def plot_residual_distribution(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual distribution (residual_deg not found)")
        return

    r = df["residual_deg"].dropna().to_numpy()
    if r.size == 0:
        print("[skip] residual distribution (empty)")
        return

    if OUTLIER_CLIP_STD is not None and r.size > 20:
        mu0 = float(np.mean(r))
        sig0 = float(np.std(r, ddof=1))
        if sig0 > 1e-12:
            keep = np.abs(r - mu0) <= OUTLIER_CLIP_STD * sig0
            r = r[keep]

    mu = float(np.mean(r))
    sig = float(np.std(r, ddof=1)) if r.size > 1 else 0.0
    med = float(np.median(r))

    df_r = pd.DataFrame({"residual_deg": r})

    fig = px.histogram(
        df_r, x="residual_deg", nbins=60, template=TEMPLATE,
        title="Residual Distribution (Innovation)",
        color_discrete_sequence=[COLORWAY[0]],
    )
    fig.update_layout(xaxis_title="residual (deg)", yaxis_title="count")
    fig.add_vline(x=mu, line_width=2, line_dash="dash",
                  annotation_text=f"mean={mu:.3f}°", annotation_position="top right")
    fig.add_vline(x=mu + sig, line_width=1, line_dash="dot")
    fig.add_vline(x=mu - sig, line_width=1, line_dash="dot")
    fig.show()

    fig2 = px.box(
        df_r, y="residual_deg", template=TEMPLATE,
        title=f"Residual Spread — median={med:.3f}°, std={sig:.3f}°",
    )
    fig2.update_layout(yaxis_title="residual (deg)")
    fig2.show()

    print("\n=== Residual Stats ===")
    print(f"Samples used : {r.size}")
    print(f"Mean         : {mu:+.6f} deg")
    print(f"Median       : {med:+.6f} deg")
    print(f"Std dev      : {sig:.6f} deg")
    print(f"Variance     : {sig*sig:.6f} deg^2")


def plot_gain_and_cov(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["K", "P"]):
        print("[skip] gain/cov (K or P not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["K"], mode="lines", name="K (Kalman gain)", line=dict(width=2)))
    if "P_pred" in df.columns:
        fig.add_trace(go.Scatter(x=df["time_s"], y=df["P_pred"], mode="lines", name="P_pred", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["P"], mode="lines", name="P (updated)", line=dict(width=2)))
    _base_layout(fig, "Kalman Gain and Covariance Evolution", "time (s)", "value")
    fig.show()
    
def plot_pred_vs_meas(df: pd.DataFrame) -> None:
    # Needs theta_pred_deg and roll_acc_deg
    if "theta_pred_deg" not in df.columns:
        print("[skip] pred vs meas (theta_pred_deg not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["theta_pred_deg"], mode="lines",
        name="theta_pred_deg (prediction)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (measurement)", line=dict(width=2),
    ))
    _base_layout(fig, "Prediction vs Measurement (tuning diagnostic)", "time (s)", "deg")
    fig.show()



def plot_ts(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["Ts_s"], mode="lines", name="Ts_s", line=dict(width=2)))
    _base_layout(fig, "Actual Sample Time Ts (dt) vs Time", "time (s)", "Ts (s)")
    fig.show()


# -----------------------------------------------------------------------------
# Main (Jupyter-safe)
# -----------------------------------------------------------------------------
def _get_csv_path() -> Path:
    """
    In Jupyter, sys.argv may contain kernel flags.
    So:
      - accept argv[1] only if it ends with '.csv'
      - else fall back to DEFAULT_PATH
    """
    if len(sys.argv) >= 2 and str(sys.argv[1]).lower().endswith(".csv"):
        return Path(sys.argv[1]).expanduser().resolve()
    return Path(DEFAULT_PATH).expanduser().resolve()


def main() -> None:
    csv_path = _get_csv_path()

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\nFix DEFAULT_PATH or pass a .csv path as argv[1]."
        )

    df = load_csv_clean(str(csv_path))

    # Skip startup transient
    if SKIP_FIRST_S and SKIP_FIRST_S > 0:
        t0 = float(df["time_s"].min())
        df = df[df["time_s"] >= (t0 + SKIP_FIRST_S)].reset_index(drop=True)

    # Ts summary
    ts = df["Ts_s"].dropna().to_numpy()
    if ts.size > 0:
        print("\n=== Ts (dt) Summary ===")
        print(f"Samples : {ts.size}")
        print(f"Mean    : {float(np.mean(ts)):.6f} s")
        if ts.size > 1:
            print(f"Std     : {float(np.std(ts, ddof=1)):.6f} s")
        print(f"Min/Max : {float(np.min(ts)):.6f} / {float(np.max(ts)):.6f} s")

    # Choose what plots you want (no more commenting/uncommenting)
    plots_to_show = [
        "roll",
        "pred_meas",
        "gain",
        "ts",
        # "gyro",
        # "acc",
        # "residual_time",
        # "residual_hist",
    ]

    if "roll" in plots_to_show:
        plot_roll_estimates(df)
    if "gyro" in plots_to_show:
        plot_gyro_rates(df)
    if "acc" in plots_to_show:
        plot_acc_components(df)
    if "residual_time" in plots_to_show:
        plot_residual_vs_time(df)
    if "residual_hist" in plots_to_show:
        plot_residual_distribution(df)
    if "gain" in plots_to_show:
        plot_gain_and_cov(df)
    if "ts" in plots_to_show:
        plot_ts(df)
    if "pred_meas" in plots_to_show:
        plot_pred_vs_meas(df)


    print("\nDone. (All figures were shown interactively.)")


if __name__ == "__main__":
    main()



=== Ts (dt) Summary ===
Samples : 1767
Mean    : 0.010000 s
Std     : 0.000000 s
Min/Max : 0.010000 / 0.010000 s



Done. (All figures were shown interactively.)


In [5]:
#!/usr/bin/env python3
# plot_kf_partc_report_plots.py
# -----------------------------------------------------------------------------
# SC651 Assignment — Part (c)/(d): Kalman Filter (gyro + accel) plotting script
#
# Robust to:
#   - SPIFFS dump markers, serial banners
#   - ---DONE--- line (serial streaming mode)
#   - corrupted/partial rows (common when serial stalls)
# Jupyter-safe + CLI-safe.
#
# CLI:
#   python3 plot_kf_partc_report_plots.py /path/to/file.csv
#
# Jupyter:
#   - Set DEFAULT_PATH below and run the cell, OR
#   - pass argv-like path by setting sys.argv manually (optional)
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: skip first few seconds (startup transients)
SKIP_FIRST_S = 1.0

# Optional: clip outliers for residual histogram
OUTLIER_CLIP_STD = 6.0

# -----------------------------
# SET THIS (JUPYTER FRIENDLY)
# -----------------------------
DEFAULT_PATH = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/run_zero_point_one.csv"


# -----------------------------------------------------------------------------
# CSV loading / cleaning
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Robust CSV loader for KF logs.

    Handles:
      - miniterm banners, dump markers
      - ---DONE--- line
      - random corrupted rows (wrong column count)
      - extra whitespace

    Strategy:
      1) Find header line starting with "time_s,"
      2) Keep only lines with the same number of comma-separated fields as header
      3) pandas read_csv on the cleaned content
      4) numeric conversion + drop rows missing time_s
    """
    p = Path(path).expanduser().resolve()
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        raw_lines = f.readlines()

    header_idx = None
    header = None
    for i, ln in enumerate(raw_lines):
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            continue
        if s.lower().startswith("time_s,"):
            header_idx = i
            header = s
            break

    if header_idx is None or header is None:
        raise ValueError(
            "Could not find CSV header line starting with 'time_s,'. "
            "Make sure your file contains the header."
        )

    n_fields = header.count(",") + 1

    cleaned = [raw_lines[header_idx].strip() + "\n"]
    for ln in raw_lines[header_idx + 1 :]:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            # skip banners, dump markers, and ---DONE---
            continue

        # Fast structural check: correct number of fields
        if (s.count(",") + 1) != n_fields:
            continue

        cleaned.append(s + "\n")

    df = pd.read_csv(StringIO("".join(cleaned)))
    df.columns = [c.strip() for c in df.columns]

    # Expected columns for the FULL logger
    expected_full = [
        "time_s", "Ts_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "ax_g", "ay_g", "az_g",
        "omega_roll_deg_s", "omega_used_deg_s",
        "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg",
        "theta_pred_deg", "residual_deg",
        "K", "P_pred", "P",
    ]

    # We allow missing columns (in case you later log a minimal set).
    # But we REQUIRE at least these:
    required_min = ["time_s", "Ts_s", "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg"]

    missing_min = [c for c in required_min if c not in df.columns]
    if missing_min:
        raise ValueError(
            f"Missing required columns: {missing_min}\nFound: {list(df.columns)}"
        )

    # Convert whatever columns exist to numeric
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop broken rows
    df = df.dropna(subset=["time_s"]).sort_values("time_s").reset_index(drop=True)

    # If residual isn't present but theta_pred is, compute it
    if "residual_deg" not in df.columns and "theta_pred_deg" in df.columns:
        df["residual_deg"] = df["roll_acc_deg"] - df["theta_pred_deg"]

    return df


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------
def _base_layout(fig: go.Figure, title: str, xlab: str, ylab: str) -> go.Figure:
    fig.update_layout(
        template=TEMPLATE,
        colorway=COLORWAY,
        title=title,
        xaxis_title=xlab,
        yaxis_title=ylab,
        hovermode="x unified",
        legend_title="signals",
    )
    return fig


def plot_roll_estimates(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"], mode="lines",
        name="roll_gyro_deg (integrated)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (accelerometer)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_kf_deg"], mode="lines",
        name="roll_kf_deg (Kalman)", line=dict(width=2),
    ))
    _base_layout(fig, "Roll Estimates vs Time", "time (s)", "roll (deg)")
    fig.show()


def plot_gyro_rates(df: pd.DataFrame) -> None:
    if "omega_roll_deg_s" not in df.columns or "omega_used_deg_s" not in df.columns:
        print("[skip] gyro rates (omega_* columns not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_roll_deg_s"], mode="lines",
        name="omega_roll_deg_s (raw)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_used_deg_s"], mode="lines",
        name="omega_used_deg_s (bias-corrected)", line=dict(width=2),
    ))
    _base_layout(fig, "Gyro Roll-Rate (raw vs bias-corrected)", "time (s)", "deg/s")
    fig.show()


def plot_acc_components(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["ax_g", "ay_g", "az_g"]):
        print("[skip] accel components (ax_g/ay_g/az_g not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ax_g"], mode="lines", name="ax_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ay_g"], mode="lines", name="ay_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["az_g"], mode="lines", name="az_g", line=dict(width=2)))
    _base_layout(fig, "Accelerometer Components vs Time", "time (s)", "acceleration (g)")
    fig.show()


def plot_residual_vs_time(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual vs time (residual_deg not found and cannot be computed)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["residual_deg"], mode="lines",
        name="residual_deg = (theta_acc - theta_pred)", line=dict(width=2),
    ))
    fig.add_hline(y=0.0, line_dash="dash", opacity=0.6)
    _base_layout(fig, "Innovation / Residual vs Time", "time (s)", "residual (deg)")
    fig.show()


def plot_residual_distribution(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual distribution (residual_deg not found)")
        return

    r = df["residual_deg"].dropna().to_numpy()
    if r.size == 0:
        print("[skip] residual distribution (empty)")
        return

    if OUTLIER_CLIP_STD is not None and r.size > 20:
        mu0 = float(np.mean(r))
        sig0 = float(np.std(r, ddof=1))
        if sig0 > 1e-12:
            keep = np.abs(r - mu0) <= OUTLIER_CLIP_STD * sig0
            r = r[keep]

    mu = float(np.mean(r))
    sig = float(np.std(r, ddof=1)) if r.size > 1 else 0.0
    med = float(np.median(r))

    df_r = pd.DataFrame({"residual_deg": r})

    fig = px.histogram(
        df_r, x="residual_deg", nbins=60, template=TEMPLATE,
        title="Residual Distribution (Innovation)",
        color_discrete_sequence=[COLORWAY[0]],
    )
    fig.update_layout(xaxis_title="residual (deg)", yaxis_title="count")
    fig.add_vline(x=mu, line_width=2, line_dash="dash",
                  annotation_text=f"mean={mu:.3f}°", annotation_position="top right")
    fig.add_vline(x=mu + sig, line_width=1, line_dash="dot")
    fig.add_vline(x=mu - sig, line_width=1, line_dash="dot")
    fig.show()

    fig2 = px.box(
        df_r, y="residual_deg", template=TEMPLATE,
        title=f"Residual Spread — median={med:.3f}°, std={sig:.3f}°",
    )
    fig2.update_layout(yaxis_title="residual (deg)")
    fig2.show()

    print("\n=== Residual Stats ===")
    print(f"Samples used : {r.size}")
    print(f"Mean         : {mu:+.6f} deg")
    print(f"Median       : {med:+.6f} deg")
    print(f"Std dev      : {sig:.6f} deg")
    print(f"Variance     : {sig*sig:.6f} deg^2")


def plot_gain_and_cov(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["K", "P"]):
        print("[skip] gain/cov (K or P not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["K"], mode="lines", name="K (Kalman gain)", line=dict(width=2)))
    if "P_pred" in df.columns:
        fig.add_trace(go.Scatter(x=df["time_s"], y=df["P_pred"], mode="lines", name="P_pred", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["P"], mode="lines", name="P (updated)", line=dict(width=2)))
    _base_layout(fig, "Kalman Gain and Covariance Evolution", "time (s)", "value")
    fig.show()
    
def plot_pred_vs_meas(df: pd.DataFrame) -> None:
    # Needs theta_pred_deg and roll_acc_deg
    if "theta_pred_deg" not in df.columns:
        print("[skip] pred vs meas (theta_pred_deg not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["theta_pred_deg"], mode="lines",
        name="theta_pred_deg (prediction)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (measurement)", line=dict(width=2),
    ))
    _base_layout(fig, "Prediction vs Measurement (tuning diagnostic)", "time (s)", "deg")
    fig.show()



def plot_ts(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["Ts_s"], mode="lines", name="Ts_s", line=dict(width=2)))
    _base_layout(fig, "Actual Sample Time Ts (dt) vs Time", "time (s)", "Ts (s)")
    fig.show()


# -----------------------------------------------------------------------------
# Main (Jupyter-safe)
# -----------------------------------------------------------------------------
def _get_csv_path() -> Path:
    """
    In Jupyter, sys.argv may contain kernel flags.
    So:
      - accept argv[1] only if it ends with '.csv'
      - else fall back to DEFAULT_PATH
    """
    if len(sys.argv) >= 2 and str(sys.argv[1]).lower().endswith(".csv"):
        return Path(sys.argv[1]).expanduser().resolve()
    return Path(DEFAULT_PATH).expanduser().resolve()


def main() -> None:
    csv_path = _get_csv_path()

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\nFix DEFAULT_PATH or pass a .csv path as argv[1]."
        )

    df = load_csv_clean(str(csv_path))

    # Skip startup transient
    if SKIP_FIRST_S and SKIP_FIRST_S > 0:
        t0 = float(df["time_s"].min())
        df = df[df["time_s"] >= (t0 + SKIP_FIRST_S)].reset_index(drop=True)

    # Ts summary
    ts = df["Ts_s"].dropna().to_numpy()
    if ts.size > 0:
        print("\n=== Ts (dt) Summary ===")
        print(f"Samples : {ts.size}")
        print(f"Mean    : {float(np.mean(ts)):.6f} s")
        if ts.size > 1:
            print(f"Std     : {float(np.std(ts, ddof=1)):.6f} s")
        print(f"Min/Max : {float(np.min(ts)):.6f} / {float(np.max(ts)):.6f} s")

    # Choose what plots you want (no more commenting/uncommenting)
    plots_to_show = [
        "roll",
        "pred_meas",
        "gain",
        "ts",
        # "gyro",
        # "acc",
        # "residual_time",
        # "residual_hist",
    ]

    if "roll" in plots_to_show:
        plot_roll_estimates(df)
    if "gyro" in plots_to_show:
        plot_gyro_rates(df)
    if "acc" in plots_to_show:
        plot_acc_components(df)
    if "residual_time" in plots_to_show:
        plot_residual_vs_time(df)
    if "residual_hist" in plots_to_show:
        plot_residual_distribution(df)
    if "gain" in plots_to_show:
        plot_gain_and_cov(df)
    if "ts" in plots_to_show:
        plot_ts(df)
    if "pred_meas" in plots_to_show:
        plot_pred_vs_meas(df)


    print("\nDone. (All figures were shown interactively.)")


if __name__ == "__main__":
    main()



=== Ts (dt) Summary ===
Samples : 1745
Mean    : 0.100000 s
Std     : 0.000000 s
Min/Max : 0.100000 / 0.100000 s



Done. (All figures were shown interactively.)


In [1]:
#!/usr/bin/env python3
# plot_kf_partc_report_plots.py
# -----------------------------------------------------------------------------
# SC651 Assignment — Part (c)/(d): Kalman Filter (gyro + accel) plotting script
#
# Robust to:
#   - SPIFFS dump markers, serial banners
#   - ---DONE--- line (serial streaming mode)
#   - corrupted/partial rows (common when serial stalls)
# Jupyter-safe + CLI-safe.
#
# CLI:
#   python3 plot_kf_partc_report_plots.py /path/to/file.csv
#
# Jupyter:
#   - Set DEFAULT_PATH below and run the cell, OR
#   - pass argv-like path by setting sys.argv manually (optional)
#
# Dependencies:
#   python3 -m pip install --user numpy pandas plotly
# -----------------------------------------------------------------------------

from __future__ import annotations

import sys
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px


TEMPLATE = "plotly_white"
COLORWAY = ["#2F80ED", "#27AE60", "#EB5757", "#9B51E0", "#F2994A", "#56CCF2"]

# Optional: skip first few seconds (startup transients)
SKIP_FIRST_S = 1.0

# Optional: clip outliers for residual histogram
OUTLIER_CLIP_STD = 6.0

# -----------------------------
# SET THIS (JUPYTER FRIENDLY)
# -----------------------------
DEFAULT_PATH = "/home/iitgn-robotics/Debojit_WS/ME692_Data_and_Observer_Design/P0_500.csv"


# -----------------------------------------------------------------------------
# CSV loading / cleaning
# -----------------------------------------------------------------------------
def load_csv_clean(path: str) -> pd.DataFrame:
    """
    Robust CSV loader for KF logs.

    Handles:
      - miniterm banners, dump markers
      - ---DONE--- line
      - random corrupted rows (wrong column count)
      - extra whitespace

    Strategy:
      1) Find header line starting with "time_s,"
      2) Keep only lines with the same number of comma-separated fields as header
      3) pandas read_csv on the cleaned content
      4) numeric conversion + drop rows missing time_s
    """
    p = Path(path).expanduser().resolve()
    with p.open("r", encoding="utf-8", errors="ignore") as f:
        raw_lines = f.readlines()

    header_idx = None
    header = None
    for i, ln in enumerate(raw_lines):
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            continue
        if s.lower().startswith("time_s,"):
            header_idx = i
            header = s
            break

    if header_idx is None or header is None:
        raise ValueError(
            "Could not find CSV header line starting with 'time_s,'. "
            "Make sure your file contains the header."
        )

    n_fields = header.count(",") + 1

    cleaned = [raw_lines[header_idx].strip() + "\n"]
    for ln in raw_lines[header_idx + 1 :]:
        s = ln.strip()
        if not s:
            continue
        if s.startswith("---"):
            # skip banners, dump markers, and ---DONE---
            continue

        # Fast structural check: correct number of fields
        if (s.count(",") + 1) != n_fields:
            continue

        cleaned.append(s + "\n")

    df = pd.read_csv(StringIO("".join(cleaned)))
    df.columns = [c.strip() for c in df.columns]

    # Expected columns for the FULL logger
    expected_full = [
        "time_s", "Ts_s",
        "gx_deg_s", "gy_deg_s", "gz_deg_s",
        "ax_g", "ay_g", "az_g",
        "omega_roll_deg_s", "omega_used_deg_s",
        "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg",
        "theta_pred_deg", "residual_deg",
        "K", "P_pred", "P",
    ]

    # We allow missing columns (in case you later log a minimal set).
    # But we REQUIRE at least these:
    required_min = ["time_s", "Ts_s", "roll_gyro_deg", "roll_acc_deg", "roll_kf_deg"]

    missing_min = [c for c in required_min if c not in df.columns]
    if missing_min:
        raise ValueError(
            f"Missing required columns: {missing_min}\nFound: {list(df.columns)}"
        )

    # Convert whatever columns exist to numeric
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Drop broken rows
    df = df.dropna(subset=["time_s"]).sort_values("time_s").reset_index(drop=True)

    # If residual isn't present but theta_pred is, compute it
    if "residual_deg" not in df.columns and "theta_pred_deg" in df.columns:
        df["residual_deg"] = df["roll_acc_deg"] - df["theta_pred_deg"]

    return df


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------
def _base_layout(fig: go.Figure, title: str, xlab: str, ylab: str) -> go.Figure:
    fig.update_layout(
        template=TEMPLATE,
        colorway=COLORWAY,
        title=title,
        xaxis_title=xlab,
        yaxis_title=ylab,
        hovermode="x unified",
        legend_title="signals",
    )
    return fig


def plot_roll_estimates(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_gyro_deg"], mode="lines",
        name="roll_gyro_deg (integrated)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (accelerometer)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_kf_deg"], mode="lines",
        name="roll_kf_deg (Kalman)", line=dict(width=2),
    ))
    _base_layout(fig, "Roll Estimates vs Time", "time (s)", "roll (deg)")
    fig.show()


def plot_gyro_rates(df: pd.DataFrame) -> None:
    if "omega_roll_deg_s" not in df.columns or "omega_used_deg_s" not in df.columns:
        print("[skip] gyro rates (omega_* columns not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_roll_deg_s"], mode="lines",
        name="omega_roll_deg_s (raw)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["omega_used_deg_s"], mode="lines",
        name="omega_used_deg_s (bias-corrected)", line=dict(width=2),
    ))
    _base_layout(fig, "Gyro Roll-Rate (raw vs bias-corrected)", "time (s)", "deg/s")
    fig.show()


def plot_acc_components(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["ax_g", "ay_g", "az_g"]):
        print("[skip] accel components (ax_g/ay_g/az_g not found)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ax_g"], mode="lines", name="ax_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["ay_g"], mode="lines", name="ay_g", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["az_g"], mode="lines", name="az_g", line=dict(width=2)))
    _base_layout(fig, "Accelerometer Components vs Time", "time (s)", "acceleration (g)")
    fig.show()


def plot_residual_vs_time(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual vs time (residual_deg not found and cannot be computed)")
        return
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["residual_deg"], mode="lines",
        name="residual_deg = (theta_acc - theta_pred)", line=dict(width=2),
    ))
    fig.add_hline(y=0.0, line_dash="dash", opacity=0.6)
    _base_layout(fig, "Innovation / Residual vs Time", "time (s)", "residual (deg)")
    fig.show()


def plot_residual_distribution(df: pd.DataFrame) -> None:
    if "residual_deg" not in df.columns:
        print("[skip] residual distribution (residual_deg not found)")
        return

    r = df["residual_deg"].dropna().to_numpy()
    if r.size == 0:
        print("[skip] residual distribution (empty)")
        return

    if OUTLIER_CLIP_STD is not None and r.size > 20:
        mu0 = float(np.mean(r))
        sig0 = float(np.std(r, ddof=1))
        if sig0 > 1e-12:
            keep = np.abs(r - mu0) <= OUTLIER_CLIP_STD * sig0
            r = r[keep]

    mu = float(np.mean(r))
    sig = float(np.std(r, ddof=1)) if r.size > 1 else 0.0
    med = float(np.median(r))

    df_r = pd.DataFrame({"residual_deg": r})

    fig = px.histogram(
        df_r, x="residual_deg", nbins=60, template=TEMPLATE,
        title="Residual Distribution (Innovation)",
        color_discrete_sequence=[COLORWAY[0]],
    )
    fig.update_layout(xaxis_title="residual (deg)", yaxis_title="count")
    fig.add_vline(x=mu, line_width=2, line_dash="dash",
                  annotation_text=f"mean={mu:.3f}°", annotation_position="top right")
    fig.add_vline(x=mu + sig, line_width=1, line_dash="dot")
    fig.add_vline(x=mu - sig, line_width=1, line_dash="dot")
    fig.show()

    fig2 = px.box(
        df_r, y="residual_deg", template=TEMPLATE,
        title=f"Residual Spread — median={med:.3f}°, std={sig:.3f}°",
    )
    fig2.update_layout(yaxis_title="residual (deg)")
    fig2.show()

    print("\n=== Residual Stats ===")
    print(f"Samples used : {r.size}")
    print(f"Mean         : {mu:+.6f} deg")
    print(f"Median       : {med:+.6f} deg")
    print(f"Std dev      : {sig:.6f} deg")
    print(f"Variance     : {sig*sig:.6f} deg^2")


def plot_gain_and_cov(df: pd.DataFrame) -> None:
    if not all(c in df.columns for c in ["K", "P"]):
        print("[skip] gain/cov (K or P not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["K"], mode="lines", name="K (Kalman gain)", line=dict(width=2)))
    if "P_pred" in df.columns:
        fig.add_trace(go.Scatter(x=df["time_s"], y=df["P_pred"], mode="lines", name="P_pred", line=dict(width=2)))
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["P"], mode="lines", name="P (updated)", line=dict(width=2)))
    _base_layout(fig, "Kalman Gain and Covariance Evolution", "time (s)", "value")
    fig.show()
    
def plot_pred_vs_meas(df: pd.DataFrame) -> None:
    # Needs theta_pred_deg and roll_acc_deg
    if "theta_pred_deg" not in df.columns:
        print("[skip] pred vs meas (theta_pred_deg not found)")
        return

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["theta_pred_deg"], mode="lines",
        name="theta_pred_deg (prediction)", line=dict(width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df["time_s"], y=df["roll_acc_deg"], mode="lines",
        name="roll_acc_deg (measurement)", line=dict(width=2),
    ))
    _base_layout(fig, "Prediction vs Measurement (tuning diagnostic)", "time (s)", "deg")
    fig.show()



def plot_ts(df: pd.DataFrame) -> None:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df["time_s"], y=df["Ts_s"], mode="lines", name="Ts_s", line=dict(width=2)))
    _base_layout(fig, "Actual Sample Time Ts (dt) vs Time", "time (s)", "Ts (s)")
    fig.show()


# -----------------------------------------------------------------------------
# Main (Jupyter-safe)
# -----------------------------------------------------------------------------
def _get_csv_path() -> Path:
    """
    In Jupyter, sys.argv may contain kernel flags.
    So:
      - accept argv[1] only if it ends with '.csv'
      - else fall back to DEFAULT_PATH
    """
    if len(sys.argv) >= 2 and str(sys.argv[1]).lower().endswith(".csv"):
        return Path(sys.argv[1]).expanduser().resolve()
    return Path(DEFAULT_PATH).expanduser().resolve()


def main() -> None:
    csv_path = _get_csv_path()

    if not csv_path.exists():
        raise FileNotFoundError(
            f"CSV not found: {csv_path}\nFix DEFAULT_PATH or pass a .csv path as argv[1]."
        )

    df = load_csv_clean(str(csv_path))

    # Skip startup transient
    if SKIP_FIRST_S and SKIP_FIRST_S > 0:
        t0 = float(df["time_s"].min())
        df = df[df["time_s"] >= (t0 + SKIP_FIRST_S)].reset_index(drop=True)

    # Ts summary
    ts = df["Ts_s"].dropna().to_numpy()
    if ts.size > 0:
        print("\n=== Ts (dt) Summary ===")
        print(f"Samples : {ts.size}")
        print(f"Mean    : {float(np.mean(ts)):.6f} s")
        if ts.size > 1:
            print(f"Std     : {float(np.std(ts, ddof=1)):.6f} s")
        print(f"Min/Max : {float(np.min(ts)):.6f} / {float(np.max(ts)):.6f} s")

    # Choose what plots you want (no more commenting/uncommenting)
    plots_to_show = [
        "roll",
        "pred_meas",
        "gain",
        "ts",
        # "gyro",
        # "acc",
        # "residual_time",
        # "residual_hist",
    ]

    if "roll" in plots_to_show:
        plot_roll_estimates(df)
    if "gyro" in plots_to_show:
        plot_gyro_rates(df)
    if "acc" in plots_to_show:
        plot_acc_components(df)
    if "residual_time" in plots_to_show:
        plot_residual_vs_time(df)
    if "residual_hist" in plots_to_show:
        plot_residual_distribution(df)
    if "gain" in plots_to_show:
        plot_gain_and_cov(df)
    if "ts" in plots_to_show:
        plot_ts(df)
    if "pred_meas" in plots_to_show:
        plot_pred_vs_meas(df)


    print("\nDone. (All figures were shown interactively.)")


if __name__ == "__main__":
    main()



=== Ts (dt) Summary ===
Samples : 1620
Mean    : 0.021000 s
Std     : 0.000002 s
Min/Max : 0.020988 / 0.021014 s



Done. (All figures were shown interactively.)
